# NileGuard — Direct Multi-Horizon Drought (PDSI) Forecasting
## v17 (`v2_optimized`) — Performance Refactor for Transfer Learning & Documentation

**Scope of this pass.** This refactor sits on top of the v16 Direct
Multi-Horizon architecture and fixes four specific production issues,
then reorganizes the entire notebook into 7 clearly documented sections:

1. **Long-horizon degradation & loss weighting** — `HorizonWeightedHuberLoss`
   now uses a steeper, power-law penalty ramp (`gamma=3.0`, `power=1.5`) so
   h=6/9/12 errors are penalized far more heavily than h=1 during training,
   on top of the existing Direct Multi-Head Output architecture (one
   forward pass → all horizons, zero autoregressive rollout — this was
   already the v16 fix for compounding drift and is preserved here).
2. **Calibration functionality & fallback** — the old global mean-offset
   calibration (which produced *negative* RMSE improvement in some
   governorates) is replaced by `HorizonSpecificCalibrator`: Isotonic
   Regression fit independently per (governorate, horizon) pair, with a
   **strict automatic fallback** — if calibration does not strictly reduce
   RMSE on its own calibration split, that pair is rejected and the raw,
   uncalibrated prediction is used instead. Calibration can never make a
   forecast worse than leaving it alone.
3. **Locked lookback window** — `LOOKBACK = 24` (months) is now a single
   global constant used by every data-loading, windowing, dataset, and
   model-input code path. The old per-governorate `LOOKBACK_MAP` (13–32
   months, varying by governorate) has been removed.
4. **Renamed artifacts** — every exported checkpoint, table, and plot
   carries the `_v2_optimized` suffix so this run never overwrites prior
   evaluation logs.

**Notebook structure (7 sections):**

| # | Section | Contents |
|---|---------|----------|
| 1 | Imports & Global Configuration | Imports, reproducibility, device, logging, locked `LOOKBACK`, directories |
| 2 | Data Loading & Preprocessing | Tensor loading/validation, feature registry, governorate bounding boxes |
| 3 | Dataset Definition & Windowing | `NileGuardGovernorateDataset`, `NileGuardDirectMultiHorizonDataset`, rolling CV folds |
| 4 | Model Architecture | CNN + Spatio-Temporal Transformer encoder, `DirectMultiHorizonDecoder` (Direct Multi-Head Output) |
| 5 | Weighted Loss & Calibration (with Fallback) | `HorizonWeightedHuberLoss`, `HorizonSpecificCalibrator`, `compute_metrics` |
| 6 | Training & Validation Loop | EMA, checkpointing, LR schedule, `train_fold_direct`, CV + final production training |
| 7 | Evaluation, Metrics & Artifact Exporters | Calibrated-vs-uncalibrated reporting, best-governorate/horizon summaries, artifact export |

**Prerequisite.** This notebook expects `nileguard_final_tensor_final.npy`
(shape `(768, 144, 72, 14, 1)`, squeezed to `(768, 144, 72, 14)`) in the
working directory — the same production tensor used by prior NileGuard
versions.


## 1. Imports, Reproducibility, Device, Logging & Global Configuration

Locks `LOOKBACK = 24` as a single global constant used everywhere else in this notebook, and sets up `_v2_optimized`-suffixed output directories.

In [1]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 1: Imports, Reproducibility,
# Device, Logging & Global Configuration
#
# This pass performs a full architectural refactor of the v16 Direct
# Multi-Horizon notebook to fix long-horizon degradation, replace the
# defective post-hoc calibration with a leakage-free, per-(governorate,
# horizon) Isotonic Regression + strict RMSE fallback, LOCK the lookback
# window at 24 months everywhere, and reorganize the entire pipeline into
# 7 clearly documented sections. All exported artifacts use the
# `_v2_optimized` suffix so they never collide with prior evaluation logs.
# =============================================================================

import sys

# Single, quiet, idempotent install step (kept for a fresh Colab/kernel).
get_ipython().system(f'{sys.executable} -m pip install -q pyyaml tensorboard ipywidgets')

import os
import gc
import math
import json
import random
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.isotonic import IsotonicRegression

# =============================================================================
# Reproducibility
# =============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =============================================================================
# Device
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================================================
# Global Lookback Lock
#
# v17 fix: earlier versions (v11-v16) resolved a DIFFERENT lookback window
# per governorate via `LOOKBACK_MAP` (e.g. 13 for Qena, 32 for Asyut). That
# meant every governorate's model saw a different amount of history, made
# cross-governorate comparisons apples-to-oranges, and was never validated
# as actually optimal per-governorate (it was an ad-hoc historical choice).
# `LOOKBACK` below is now a SINGLE, EXPLICIT, LOCKED constant used by every
# data-loading, windowing, dataset, and model-input code path in this
# notebook. There is exactly one place to change it.
# =============================================================================
LOOKBACK = 24  # months — LOCKED. Do not override per-governorate.

# Direct multi-horizon forecast targets (months ahead), longest-horizon-last.
HORIZONS_DIRECT = [1, 3, 6, 9, 12]

# =============================================================================
# Project Directories (v2_optimized artifacts — never overwrite prior runs)
# =============================================================================
PROJECT_DIR = Path(".")
RESULTS_DIR = PROJECT_DIR / "results_v2_optimized"
LOGS_DIR = PROJECT_DIR / "logs_v2_optimized"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints_v2_optimized"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts_v2_optimized"
TENSORBOARD_DIR = LOGS_DIR / "tensorboard_v2_optimized"

for directory in [RESULTS_DIR, LOGS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR, TENSORBOARD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Logging
# =============================================================================
def setup_logger(
    name: str = "NileGuard",
    log_file: str = str(LOGS_DIR / "nileguard_v2_optimized.log"),
    level: int = logging.INFO,
) -> logging.Logger:
    """Creates (or returns) a singleton console+file logger.

    Args:
        name: Logger name. Reusing the same name returns the same
            logger instance instead of duplicating handlers.
        log_file: Path to the log file. If the path is not writable
            (e.g. a read-only filesystem), file logging is silently
            skipped and console logging still works.
        level: Logging verbosity threshold (default ``logging.INFO``).

    Returns:
        A configured ``logging.Logger`` instance.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False

    if not logger.handlers:
        fmt = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            "%Y-%m-%d %H:%M:%S",
        )
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(fmt)
        logger.addHandler(stream_handler)
        try:
            file_handler = logging.FileHandler(log_file)
            file_handler.setFormatter(fmt)
            logger.addHandler(file_handler)
        except OSError:
            pass

    return logger


logger = setup_logger()

logger.info("=" * 78)
logger.info("NileGuard v2_optimized — Direct Multi-Horizon Drought (PDSI) Forecasting")
logger.info("=" * 78)
logger.info(f"Using device       : {DEVICE}")
logger.info(f"PyTorch version    : {torch.__version__}")
logger.info(f"LOCKED lookback    : {LOOKBACK} months")
logger.info(f"Forecast horizons  : {HORIZONS_DIRECT} months")
logger.info(f"Random seed        : {SEED}")
logger.info(f"Results directory  : {RESULTS_DIR}")
logger.info(f"Artifacts directory: {ARTIFACTS_DIR}")



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
2026-08-23 19:53:00 | INFO     | NileGuard | ==============================================================================
2026-08-23 19:53:00 | INFO     | NileGuard | NileGuard v2_optimized — Direct Multi-Horizon Drought (PDSI) Forecasting
2026-08-23 19:53:00 | INFO     | NileGuard | ==============================================================================
2026-08-23 19:53:00 | INFO     | NileGuard | Using device       : cuda
2026-08-23 19:53:00 | INFO     | NileGuard | PyTorch version    : 2.5.1+cu121
2026-08-23 19:53:00 | INFO     | NileGuard | LOCKED lookback    : 24 months
2026-08-23 19:53:00 | INFO     | NileGuard | Forecast horizons  : [1, 3, 6, 9, 12] months
2026-08-23 19:53:00 | INFO     | NileGuard | Random seed        : 42
2026-08-23 19:53:00 | INFO     | NileGuard | Results directory  : results_v2_optimized
2026-08-23 19:53:00 | INFO     | Nil

## 2. Data Loading & Preprocessing

Loads and validates `master_tensor`, builds the feature registry, and computes each governorate's real (irregular) bounding box.

In [2]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 2: Data Loading & Preprocessing
#
# Loads the master 5-D climate tensor, coerces it to (T, H, W, C), verifies
# the target (PDSI) and governorate-mask channels, then computes each
# governorate's real (irregular) pixel footprint as a tight bounding box.
# =============================================================================

DATA_PATH = "nileguard_final_tensor_final.npy"

raw = np.load(DATA_PATH)
logger.info(f"Raw file shape : {raw.shape}")

# The file is stored as (T, H, W, C, 1). Remove the trailing singleton axis.
if raw.ndim == 5 and raw.shape[-1] == 1:
    master_tensor = np.squeeze(raw, axis=-1).astype(np.float32)
else:
    master_tensor = raw.astype(np.float32)

# Production tensor: 13 active climate variables (SWE dropped upstream -
# all-zero across Egypt) + 1 governorate-ID channel = 14 channels.
assert master_tensor.shape == (768, 144, 72, 14), (
    f"master_tensor has shape {master_tensor.shape}, expected (768, 144, 72, 14)"
)
assert not np.isnan(master_tensor).any(), "master_tensor contains NaNs"

T_TOTAL, H, W, C = master_tensor.shape
logger.info(f"Master Tensor Loaded : shape={master_tensor.shape}, T={T_TOTAL}, H={H}, W={W}, Channels={C}")

# =============================================================================
# Feature Registry
# =============================================================================
FEATURE_NAMES = [
    "aet", "def", "PDSI", "pet", "ppt", "q", "soil", "srad",
    "tmax", "tmin", "vap", "vpd", "ws", "governorate",
]
assert len(FEATURE_NAMES) == C, (
    f"FEATURE_NAMES has {len(FEATURE_NAMES)} entries but master_tensor has {C} channels - "
    "these must match exactly or every downstream index will silently point at the wrong channel."
)

TARGET_CHANNEL_INDEX = FEATURE_NAMES.index("PDSI")
GOV_MASK_CHANNEL_INDEX = FEATURE_NAMES.index("governorate")
logger.info(f"PDSI target channel      : {TARGET_CHANNEL_INDEX}")
logger.info(f"Governorate mask channel : {GOV_MASK_CHANNEL_INDEX}")

# =============================================================================
# Verify Governorate Mask (static) and Target Channel (time-varying)
# =============================================================================
gov_mask_t0 = master_tensor[0, :, :, GOV_MASK_CHANNEL_INDEX]
gov_mask_t100 = master_tensor[100, :, :, GOV_MASK_CHANNEL_INDEX]
assert np.array_equal(gov_mask_t0, gov_mask_t100), f"Channel {GOV_MASK_CHANNEL_INDEX} is not static across time."
assert set(np.unique(gov_mask_t0).astype(int)) == set(range(9)), "Governorate mask should contain IDs {0,...,8}."

target_t0 = master_tensor[0, :, :, TARGET_CHANNEL_INDEX]
target_t100 = master_tensor[100, :, :, TARGET_CHANNEL_INDEX]
assert not np.array_equal(target_t0, target_t100), f"Target channel {TARGET_CHANNEL_INDEX} appears to be static."
logger.info("Validation passed: target channel varies in time, governorate mask is static.")

# =============================================================================
# Governorate Registry & Real Bounding Boxes
# =============================================================================
N_GOVERNORATES = 8
GOVERNORATE_NAMES = {
    1: "Aswan", 2: "Asyut", 3: "BeniSuef", 4: "Fayoum",
    5: "Luxor", 6: "Minya", 7: "Qena", 8: "Sohag",
}


def resolve_lookback(governorate_id: int, min_val_samples: int = 2) -> int:
    """Returns the LOCKED lookback window for any governorate.

    v17: replaces the old per-governorate `LOOKBACK_MAP` lookup. Every
    governorate now uses the exact same global `LOOKBACK` constant
    (Section 1), so this function's only remaining job is to validate
    that the locked value is actually usable against the dataset's total
    time length, raising a clear error instead of a confusing downstream
    shape/empty-fold failure.

    Args:
        governorate_id: Governorate ID (1-8). Retained as a parameter
            for call-site compatibility even though the resolved value
            no longer varies by governorate.
        min_val_samples: Minimum number of usable validation samples the
            locked lookback must leave behind.

    Returns:
        The locked lookback window, in months.

    Raises:
        ValueError: If ``LOOKBACK`` is non-positive or leaves fewer than
            ``min_val_samples`` usable samples given ``T_TOTAL``.
    """
    lookback = int(LOOKBACK)
    if lookback < 1:
        raise ValueError(f"LOOKBACK={lookback} must be a positive integer.")
    if "T_TOTAL" in globals():
        max_usable = T_TOTAL - min_val_samples
        if lookback >= max_usable:
            raise ValueError(
                f"LOOKBACK={lookback} leaves fewer than {min_val_samples} usable samples "
                f"out of T_TOTAL={T_TOTAL}. Reduce LOOKBACK or provide more history."
            )
    return lookback


def compute_governorate_bbox(gov_mask: np.ndarray, gov_id: int):
    """Computes the tight bounding box enclosing a governorate's pixels.

    Args:
        gov_mask: (H, W) array of governorate IDs.
        gov_id: The governorate ID to locate.

    Returns:
        A ``(row_start, row_end, col_start, col_end)`` tuple, with both
        ends exclusive on the upper bound (NumPy slice convention).

    Raises:
        ValueError: If ``gov_id`` has zero pixels in ``gov_mask``.
    """
    rows, cols = np.where(gov_mask == gov_id)
    if len(rows) == 0:
        raise ValueError(f"Governorate ID {gov_id} has zero pixels in the mask.")
    return rows.min(), rows.max() + 1, cols.min(), cols.max() + 1


GOV_BBOXES = {}
for gov_id, gov_name in GOVERNORATE_NAMES.items():
    r0, r1, c0, c1 = compute_governorate_bbox(gov_mask_t0, gov_id)
    GOV_BBOXES[gov_id] = (r0, r1, c0, c1)
    n_pixels = int((gov_mask_t0 == gov_id).sum())
    logger.info(
        f"Governorate {gov_name:<9} (id={gov_id}) | pixels={n_pixels:>6} | "
        f"bbox={r1 - r0}x{c1 - c0} | lookback={LOOKBACK} (locked)"
    )


def compute_norm_stats(master_tensor: np.ndarray, train_end_idx: int):
    """Computes per-channel mean/std from a training-only time slice.

    Args:
        master_tensor: (T, H, W, C) tensor.
        train_end_idx: Exclusive end index of the training window; only
            ``master_tensor[:train_end_idx]`` is used, so validation/test
            statistics never leak into normalization.

    Returns:
        A ``(mean, std)`` tuple of ``(C,)`` float32 arrays. Any channel
        with near-zero std is clamped to std=1.0 to avoid division by
        zero (that channel becomes a simple mean-centered value).
    """
    train_slice = master_tensor[:train_end_idx]
    mean = train_slice.mean(axis=(0, 1, 2))
    std = train_slice.std(axis=(0, 1, 2))
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


2026-08-23 19:53:03 | INFO     | NileGuard | Raw file shape : (768, 144, 72, 14, 1)
2026-08-23 19:53:03 | INFO     | NileGuard | Master Tensor Loaded : shape=(768, 144, 72, 14), T=768, H=144, W=72, Channels=14
2026-08-23 19:53:03 | INFO     | NileGuard | PDSI target channel      : 2
2026-08-23 19:53:03 | INFO     | NileGuard | Governorate mask channel : 13
2026-08-23 19:53:03 | INFO     | NileGuard | Validation passed: target channel varies in time, governorate mask is static.
2026-08-23 19:53:03 | INFO     | NileGuard | Governorate Aswan     (id=1) | pixels=   140 | bbox=14x17 | lookback=24 (locked)
2026-08-23 19:53:03 | INFO     | NileGuard | Governorate Asyut     (id=2) | pixels=   732 | bbox=22x56 | lookback=24 (locked)
2026-08-23 19:53:03 | INFO     | NileGuard | Governorate BeniSuef  (id=3) | pixels=    14 | bbox=5x5 | lookback=24 (locked)
2026-08-23 19:53:03 | INFO     | NileGuard | Governorate Fayoum    (id=4) | pixels=  1377 | bbox=64x35 | lookback=24 (locked)
2026-08-23 19:53

## 3. Dataset Definition & Windowing

`NileGuardGovernorateDataset` (single-step crop/normalize contract, reused internally) and `NileGuardDirectMultiHorizonDataset` (real targets at every horizon in one sample), plus the rolling/expanding chronological CV-fold generator. Both datasets use the LOCKED `LOOKBACK` constant.

In [3]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 3: Dataset Definition & Windowing
#
# Every dataset and CV-fold generator below uses the LOCKED `LOOKBACK`
# constant from Section 1. No code path in this notebook accepts a
# per-call lookback override anymore.
# =============================================================================

class NileGuardGovernorateDataset(Dataset):
    """Crops, normalizes, and windows one governorate's climate region.

    Crops `master_tensor` to a governorate's real bounding box, normalizes
    the 13 active climate channels with the supplied ``channel_mean`` /
    ``channel_std``, and replaces the raw governorate-ID channel with a
    clean binary in-governorate mask feature.

    Each sample is:
        X: (LOOKBACK, C, H_gov, W_gov)
        y: (H_gov, W_gov) — DELTA target = normalized(t) - normalized(t-1)

    Because both frames are z-scored with the same per-fold
    ``(channel_mean, channel_std)``, the target mean cancels exactly in
    the subtraction, so ``y_delta * target_std`` recovers the raw ΔPDSI
    in physical units directly.
    """

    def __init__(self, master_tensor: np.ndarray, governorate_id: int, lookback: int,
                 channel_mean: np.ndarray, channel_std: np.ndarray):
        """Builds the normalized, cropped region tensor for one governorate.

        Args:
            master_tensor: (T, H, W, C) raw climate tensor.
            governorate_id: Governorate ID (1-8); must be a key of
                ``GOV_BBOXES``.
            lookback: Number of past months per input window. Must equal
                the LOCKED ``LOOKBACK`` constant at every call site in
                this notebook.
            channel_mean: (C,) per-channel training-set mean.
            channel_std: (C,) per-channel training-set std (all > 0).

        Raises:
            ValueError: On an unknown governorate ID, non-positive
                lookback, wrong tensor rank, mismatched normalization
                stat shapes, non-positive std, too-short timeline, a
                fully-empty governorate crop, or non-finite values after
                normalization.
        """
        if governorate_id not in GOV_BBOXES:
            raise ValueError(f"Unknown governorate_id {governorate_id}; valid ids: {sorted(GOV_BBOXES)}")
        if not isinstance(lookback, (int, np.integer)) or lookback < 1:
            raise ValueError(f"lookback must be a positive integer, got {lookback!r}")
        if master_tensor.ndim != 4:
            raise ValueError(f"master_tensor must be 4-D (T, H, W, C), got shape {master_tensor.shape}")

        r0, r1, c0, c1 = GOV_BBOXES[governorate_id]
        n_channels = master_tensor.shape[-1]
        channel_mean = np.asarray(channel_mean, dtype=np.float32)
        channel_std = np.asarray(channel_std, dtype=np.float32)
        if channel_mean.shape != (n_channels,) or channel_std.shape != (n_channels,):
            raise ValueError(
                f"channel_mean/channel_std must each have shape ({n_channels},); "
                f"got {channel_mean.shape} / {channel_std.shape}"
            )
        if np.any(channel_std <= 0):
            raise ValueError("channel_std contains zero or negative values - normalization would be invalid.")
        if master_tensor.shape[0] <= lookback:
            raise ValueError(
                f"lookback={lookback} leaves zero usable samples for a timeline of "
                f"length {master_tensor.shape[0]} (need T > lookback)."
            )

        region = master_tensor[:, r0:r1, c0:c1, :].astype(np.float32, copy=True)  # (T, Hg, Wg, C)

        with np.errstate(invalid="raise", divide="raise"):
            norm_mask = np.ones(n_channels, dtype=bool)
            norm_mask[GOV_MASK_CHANNEL_INDEX] = False
            region[..., norm_mask] = (region[..., norm_mask] - channel_mean[norm_mask]) / channel_std[norm_mask]

        if not np.isfinite(region[..., norm_mask]).all():
            raise ValueError(
                f"Non-finite values detected after normalization for governorate {governorate_id} "
                "- check channel_mean/channel_std and the source tensor for NaN/Inf."
            )

        gov_crop_mask = (master_tensor[0, r0:r1, c0:c1, GOV_MASK_CHANNEL_INDEX] == governorate_id)
        if not gov_crop_mask.any():
            raise ValueError(f"Governorate {governorate_id} bounding-box crop contains zero real pixels.")
        region[..., GOV_MASK_CHANNEL_INDEX] = gov_crop_mask.astype(np.float32)[None, :, :]

        region = np.ascontiguousarray(np.transpose(region, (0, 3, 1, 2)))  # (T, C, Hg, Wg)

        self.region = torch.from_numpy(region)
        self.target_mask = torch.from_numpy(gov_crop_mask).bool()  # (Hg, Wg)
        self.governorate_id = governorate_id
        self.lookback = int(lookback)
        self.channel_mean = channel_mean
        self.channel_std = channel_std
        self.T, self.C, self.H_gov, self.W_gov = self.region.shape

    def __len__(self):
        return max(0, self.T - self.lookback)

    def __getitem__(self, idx):
        if idx < 0 or idx >= len(self):
            raise IndexError(f"Sample index {idx} out of range for dataset of length {len(self)}")
        X = self.region[idx: idx + self.lookback]
        last_frame = self.region[idx + self.lookback - 1, TARGET_CHANNEL_INDEX]
        absolute_next = self.region[idx + self.lookback, TARGET_CHANNEL_INDEX]
        y_delta = absolute_next - last_frame
        return X, y_delta


class NileGuardDirectMultiHorizonDataset(Dataset):
    """Direct multi-horizon dataset: one sample yields REAL targets for every horizon.

    Reuses ``NileGuardGovernorateDataset`` internally for the exact same
    governorate crop / normalization contract, so there is exactly one
    source of truth for how a governorate's region tensor is built.

    ``__getitem__`` returns:
        X       : (LOOKBACK, C, Hg, Wg)
        y_delta : (n_horizons, Hg, Wg) — REAL normalized deltas at every
            horizon, each computed directly from a real future frame
            (``y_delta[i] = region[t + h_i] - region[t]``), never from
            another horizon's prediction.

    Only sample-start indices with the MAX horizon's real future frame
    available are usable, so every horizon head trains/evaluates on
    exactly the same sample count.
    """

    def __init__(self, master_tensor: np.ndarray, governorate_id: int, lookback: int,
                 channel_mean: np.ndarray, channel_std: np.ndarray, horizons: list = None):
        """Builds the multi-horizon dataset for one governorate.

        Args:
            master_tensor: (T, H, W, C) raw climate tensor.
            governorate_id: Governorate ID (1-8).
            lookback: Input window length in months (must equal the
                LOCKED ``LOOKBACK`` constant).
            channel_mean: (C,) per-channel training-set mean.
            channel_std: (C,) per-channel training-set std.
            horizons: List of forecast horizons in months. Defaults to
                ``HORIZONS_DIRECT`` (``[1, 3, 6, 9, 12]``).
        """
        self.horizons = sorted(horizons if horizons is not None else HORIZONS_DIRECT)
        self.max_h = max(self.horizons)

        base = NileGuardGovernorateDataset(master_tensor, governorate_id, lookback, channel_mean, channel_std)
        self.region = base.region
        self.target_mask = base.target_mask
        self.governorate_id = governorate_id
        self.lookback = int(lookback)
        self.channel_mean = base.channel_mean
        self.channel_std = base.channel_std
        self.T, self.C, self.H_gov, self.W_gov = self.region.shape

    def __len__(self):
        return max(0, self.T - self.lookback - self.max_h + 1)

    def __getitem__(self, idx):
        if idx < 0 or idx >= len(self):
            raise IndexError(f"Sample index {idx} out of range for dataset of length {len(self)}")
        X = self.region[idx: idx + self.lookback]
        last_frame = self.region[idx + self.lookback - 1, TARGET_CHANNEL_INDEX]
        deltas = []
        for h in self.horizons:
            future_idx = idx + self.lookback - 1 + h
            future_frame = self.region[future_idx, TARGET_CHANNEL_INDEX]
            deltas.append(future_frame - last_frame)
        y_delta = torch.stack(deltas, dim=0)
        return X, y_delta


# =============================================================================
# Moving / Rolling Cross-Validation — chronological, leakage-free fold generator
# =============================================================================
N_CV_FOLDS = 4
CV_MIN_TRAIN_FRACTION = 0.55
CV_VAL_FRACTION = 0.10
CV_MODE = "expanding"
CV_GAP_SAMPLES = 0


def generate_rolling_folds(n_samples: int, n_folds: int = N_CV_FOLDS,
                            min_train_fraction: float = CV_MIN_TRAIN_FRACTION,
                            val_fraction: float = CV_VAL_FRACTION,
                            mode: str = CV_MODE, gap: int = CV_GAP_SAMPLES):
    """Generates rolling/expanding chronological CV folds.

    Args:
        n_samples: Number of chronologically ordered sample-start indices.
        n_folds: Number of folds to generate.
        min_train_fraction: Minimum fraction of ``n_samples`` reserved
            for training in every fold.
        val_fraction: Fraction of ``n_samples`` used for validation in
            every fold.
        mode: ``"expanding"`` (train always starts at 0) or
            ``"sliding"`` (train window slides forward, fixed length).
        gap: Samples purged between train end and val start. Should be
            set to at least ``LOOKBACK`` (plus any max horizon, where
            applicable) to guarantee zero leakage between a fold's
            training and validation input windows.

    Returns:
        A list of ``(train_idx, val_idx)`` tuples of 1-D ``np.ndarray``
        sample-start indices, chronologically ordered.

    Raises:
        ValueError: If ``gap`` is negative or there are not enough
            samples to build any leakage-free fold under the given
            fractions.
    """
    if gap < 0:
        raise ValueError(f"gap must be >= 0, got {gap}.")

    val_size = max(1, int(round(n_samples * val_fraction)))
    min_train_size = max(1, int(round(n_samples * min_train_fraction)))

    max_train_end = n_samples - val_size - gap
    if max_train_end <= min_train_size:
        raise ValueError(
            f"Not enough samples ({n_samples}) to build {n_folds} rolling folds "
            f"with min_train_fraction={min_train_fraction}, val_fraction={val_fraction}, gap={gap}."
        )

    train_ends = (
        [max_train_end] if n_folds == 1
        else np.linspace(min_train_size, max_train_end, n_folds).astype(int).tolist()
    )

    folds = []
    for train_end in sorted(set(train_ends)):
        val_start = train_end + gap
        val_end = min(val_start + val_size, n_samples)
        if val_end <= val_start:
            continue
        train_start = max(0, train_end - min_train_size) if mode == "sliding" else 0
        folds.append((np.arange(train_start, train_end), np.arange(val_start, val_end)))
    return folds


# ── Smoke tests: confirm the locked lookback flows through both datasets ───────────
_demo_mean, _demo_std = compute_norm_stats(master_tensor, train_end_idx=int(T_TOTAL * 0.8))
for gov_id in range(1, N_GOVERNORATES + 1):
    ds = NileGuardDirectMultiHorizonDataset(
        master_tensor, gov_id, resolve_lookback(gov_id), _demo_mean, _demo_std, horizons=HORIZONS_DIRECT
    )
    X, y = ds[0]
    logger.info(
        f"Governorate {GOVERNORATE_NAMES[gov_id]:<9} -> dataset size: {len(ds)}, "
        f"X: {tuple(X.shape)} (lookback={ds.lookback}), y_delta: {tuple(y.shape)} "
        f"(horizons={ds.horizons}), real pixels: {int(ds.target_mask.sum())}"
    )
del ds, X, y
gc.collect()


2026-08-23 19:53:03 | INFO     | NileGuard | Governorate Aswan     -> dataset size: 733, X: (24, 14, 14, 17) (lookback=24), y_delta: (5, 14, 17) (horizons=[1, 3, 6, 9, 12]), real pixels: 140
2026-08-23 19:53:04 | INFO     | NileGuard | Governorate Asyut     -> dataset size: 733, X: (24, 14, 22, 56) (lookback=24), y_delta: (5, 22, 56) (horizons=[1, 3, 6, 9, 12]), real pixels: 732
2026-08-23 19:53:04 | INFO     | NileGuard | Governorate BeniSuef  -> dataset size: 733, X: (24, 14, 5, 5) (lookback=24), y_delta: (5, 5, 5) (horizons=[1, 3, 6, 9, 12]), real pixels: 14
2026-08-23 19:53:04 | INFO     | NileGuard | Governorate Fayoum    -> dataset size: 733, X: (24, 14, 64, 35) (lookback=24), y_delta: (5, 64, 35) (horizons=[1, 3, 6, 9, 12]), real pixels: 1377
2026-08-23 19:53:04 | INFO     | NileGuard | Governorate Luxor     -> dataset size: 733, X: (24, 14, 19, 30) (lookback=24), y_delta: (5, 19, 30) (horizons=[1, 3, 6, 9, 12]), real pixels: 384
2026-08-23 19:53:04 | INFO     | NileGuard | Gove

0

## 4. Hybrid CNN-Transformer Model Architecture (Direct Multi-Head Output)

Depthwise-Separable Multi-Scale CNN → SE Block → Feature Fusion → Spatio-Temporal Attention → Divided Space-Time Transformer → `DirectMultiHorizonDecoder`, which produces forecasts for every horizon (h=1,3,6,9,12) in a single forward pass — no autoregressive rollout, so there is no mechanism for compounding drift on long horizons.

In [4]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 4: Hybrid CNN-Transformer Model
# Architecture (Direct Multi-Head Output)
#
# Pipeline: Climate Tensor -> Depthwise-Separable Multi-Scale CNN -> SE
# Block -> Feature Fusion -> Spatio-Temporal Attention -> pooled spatial
# token grid -> Learnable Positional Embedding -> Divided Space-Time
# Transformer blocks -> temporal-attention pooling -> shared decoder grid
# -> DIRECT, per-horizon output heads (h=1,3,6,9,12 simultaneously).
#
# v17 fix: this architecture uses a Direct Multi-Head Output (one forward
# pass -> all horizons) instead of an autoregressive rollout. There is no
# mechanism anywhere in this notebook for a prediction to be fed back into
# the model as if it were a real observation, which is what previously
# caused negative R2/NSE on long horizons (compounding rollout drift).
# =============================================================================

class DepthwiseSeparableConv2d(nn.Module):
    """Depthwise (per-channel) 2-D conv followed by a 1x1 pointwise conv.

    Reduces parameter count roughly from ``k*k*Cin*Cout`` to
    ``k*k*Cin + Cin*Cout`` versus a standard ``nn.Conv2d``, while keeping
    the same receptive field.
    """

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3,
                 padding: int = 1, dilation: int = 1, bias: bool = True):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size, padding=padding,
            dilation=dilation, groups=in_channels, bias=bias,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pointwise(self.depthwise(x))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel-attention block."""

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1), nn.SiLU(),
            nn.Conv2d(hidden, channels, kernel_size=1), nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.fc(self.pool(x))
        return x * gate


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(rms + self.eps)
        return x_normed * self.weight


class DropPath(nn.Module):
    """Stochastic Depth (Huang et al., 2016): drops entire per-sample residual branches."""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class LearnablePositionalEmbedding(nn.Module):
    """Fully learnable positional embedding, broadcast over all leading dims but the last two."""

    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(-2), :]


class MultiHeadTemporalAttention(nn.Module):
    """Pre-LN Multi-Head Self-Attention across the temporal axis, applied per spatial token.

    Input/output shape: (B, S, T, d_model). Caches the most recent
    (head-averaged) attention weights in ``self.last_attn_weights``
    (shape (B, S, T, T)) for explainability.
    """

    def __init__(self, d_model: int, nhead: int, dropout: float = 0.1):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.last_attn_weights = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, T, D = x.shape
        residual = x
        x_norm = self.norm(x).reshape(B * S, T, D)
        attn_out, attn_weights = self.attn(
            x_norm, x_norm, x_norm, need_weights=True, average_attn_weights=True
        )
        self.last_attn_weights = attn_weights.reshape(B, S, T, T).detach()
        attn_out = attn_out.reshape(B, S, T, D)
        return residual + self.drop(attn_out)


class MultiScaleConvBlockV8(nn.Module):
    """Multi-scale CNN block: parallel 3x3 / 5x5 / dilated-3x3 branches, fused + residual."""

    def __init__(self, in_channels: int, out_channels: int, dropout: float):
        super().__init__()
        branch_channels = out_channels // 3
        remainder = out_channels - branch_channels * 3

        self.branch3 = nn.Sequential(
            DepthwiseSeparableConv2d(in_channels, branch_channels + remainder, kernel_size=3, padding=1),
            nn.BatchNorm2d(branch_channels + remainder), nn.SiLU(),
        )
        self.branch5 = nn.Sequential(
            DepthwiseSeparableConv2d(in_channels, branch_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(branch_channels), nn.SiLU(),
        )
        self.branch_dilated = nn.Sequential(
            DepthwiseSeparableConv2d(in_channels, branch_channels, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(branch_channels), nn.SiLU(),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels), nn.SiLU(),
        )
        self.drop = nn.Dropout2d(dropout)
        self.skip = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.skip(x)
        branches = torch.cat([self.branch3(x), self.branch5(x), self.branch_dilated(x)], dim=1)
        out = self.drop(self.fuse(branches))
        return F.silu(out + identity)


class FeatureFusionModule(nn.Module):
    """Learnable local/global feature fusion block, blended via a learnable sigmoid gate."""

    def __init__(self, channels: int, dropout: float):
        super().__init__()
        self.local_branch = nn.Sequential(
            DepthwiseSeparableConv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels), nn.SiLU(),
        )
        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels, kernel_size=1), nn.SiLU(),
            nn.Conv2d(channels, channels, kernel_size=1), nn.Sigmoid(),
        )
        self.fusion_gate = nn.Parameter(torch.tensor(0.0))
        self.drop = nn.Dropout2d(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        local_feats = self.local_branch(x)
        global_gate = self.global_branch(x)
        locally_fused = local_feats * global_gate + x * (1.0 - global_gate)
        gate = torch.sigmoid(self.fusion_gate)
        out = gate * locally_fused + (1.0 - gate) * x
        return self.drop(out)


class SpatioTemporalAttention(nn.Module):
    """Joint channel / spatial (CBAM-style) / temporal attention, each with an independent gate.

    Caches ``self.last_spatial_attn`` (B, T, H, W) for explainability.
    """

    def __init__(self, channels: int, reduction: int = 8, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.channel_fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.SiLU(),
            nn.Linear(hidden, channels), nn.Sigmoid(),
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.temporal_fc = nn.Sequential(
            nn.Linear(channels, hidden), nn.SiLU(),
            nn.Linear(hidden, 1),
        )
        self.gate_c = nn.Parameter(torch.tensor(0.0))
        self.gate_s = nn.Parameter(torch.tensor(0.0))
        self.gate_t = nn.Parameter(torch.tensor(0.0))
        self.drop = nn.Dropout(dropout)
        self.last_spatial_attn = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape
        xf = x.reshape(B * T, C, H, W)

        channel_pooled = xf.mean(dim=(2, 3))
        channel_weight = self.channel_fc(channel_pooled).view(B * T, C, 1, 1)
        gate_c = torch.sigmoid(self.gate_c)
        xc = xf * (1.0 - gate_c) + xf * channel_weight * gate_c

        avg_map = xc.mean(dim=1, keepdim=True)
        max_map, _ = xc.max(dim=1, keepdim=True)
        spatial_weight = torch.sigmoid(self.spatial_conv(torch.cat([avg_map, max_map], dim=1)))
        gate_s = torch.sigmoid(self.gate_s)
        xs = xc * (1.0 - gate_s) + xc * spatial_weight * gate_s
        self.last_spatial_attn = spatial_weight.reshape(B, T, H, W).detach()

        xs_t = xs.reshape(B, T, C, H, W)
        frame_descriptor = xs_t.mean(dim=(3, 4))
        temporal_scores = self.temporal_fc(frame_descriptor)
        temporal_weight = torch.softmax(temporal_scores, dim=1).unsqueeze(-1).unsqueeze(-1)
        gate_t = torch.sigmoid(self.gate_t)
        out = xs_t * (1.0 - gate_t) + xs_t * (temporal_weight * T) * gate_t

        return self.drop(out)


class DividedSpaceTimeBlockV8(nn.Module):
    """One block of factorized (divided) space-time self-attention over (B, S, T, d_model) tokens."""

    def __init__(self, d_model: int, nhead: int, dim_feedforward: int,
                 dropout: float, drop_path_rate: float = 0.0):
        super().__init__()
        self.temporal_attn_module = MultiHeadTemporalAttention(d_model, nhead, dropout)
        self.spatial_norm = RMSNorm(d_model)
        self.spatial_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ff_norm = RMSNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.drop_path = DropPath(drop_path_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, T, D = x.shape

        x = self.temporal_attn_module(x)

        residual = x
        xs = self.spatial_norm(x).permute(0, 2, 1, 3).reshape(B * T, S, D)
        attn_out, _ = self.spatial_attn(xs, xs, xs, need_weights=False)
        attn_out = attn_out.reshape(B, T, S, D).permute(0, 2, 1, 3)
        x = residual + self.drop_path(attn_out)

        residual = x
        x = residual + self.drop_path(self.ff(self.ff_norm(x)))
        return x


class DirectMultiHorizonDecoder(nn.Module):
    """Produces PDSI delta forecasts for every horizon in ONE forward pass.

    Each horizon head produces an INCREMENT map from the shared feature
    grid plus its own learned "how much further out is this horizon"
    embedding (FiLM conditioning). The per-horizon delta returned is the
    CUMULATIVE SUM of increments up to and including that horizon:

        delta_h1 = increment_1
        delta_h3 = increment_1 + increment_2
        delta_h6 = increment_1 + increment_2 + increment_3
        ...

    so every longer-horizon forecast is anchored on top of the
    shorter-horizon ones, without ever feeding a prediction back through
    the model (no autoregressive rollout). The final increment head is
    zero-initialized: at initialization every horizon predicts delta=0
    (the persistence baseline) and only learns positive refinements from
    there.
    """

    def __init__(self, decoder_channels: int, horizons: list, dropout: float):
        super().__init__()
        self.horizons = list(horizons)
        self.n_horizons = len(horizons)
        self.horizon_embed = nn.Parameter(torch.randn(self.n_horizons, decoder_channels) * 0.02)
        self.trunk = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(decoder_channels, decoder_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(decoder_channels),
            nn.SiLU(),
            nn.Dropout2d(dropout),
        )
        self.increment_head = nn.Conv2d(decoder_channels, 1, kernel_size=1)
        nn.init.zeros_(self.increment_head.weight)
        nn.init.zeros_(self.increment_head.bias)

    def forward(self, grid: torch.Tensor) -> torch.Tensor:
        """Args: grid (B, decoder_channels, H, W). Returns: (B, n_horizons, H, W) cumulative deltas."""
        increments = []
        for i in range(self.n_horizons):
            film = self.horizon_embed[i].view(1, -1, 1, 1)
            feat = self.trunk(grid + film)
            inc = self.increment_head(feat).squeeze(1)
            increments.append(inc)
        increments = torch.stack(increments, dim=1)
        cumulative_deltas = torch.cumsum(increments, dim=1)
        return cumulative_deltas


class NileGuardDirectMultiHorizonModel(nn.Module):
    """v17 — Direct multi-horizon PDSI forecaster (Direct Multi-Head Output).

    Encoder: Depthwise-Separable Multi-Scale CNN -> SE Block -> Feature
    Fusion -> Spatio-Temporal Attention -> pooled spatial-token grid ->
    Learnable Positional Embedding -> Divided Space-Time Transformer
    blocks -> temporal-attention pooling -> decoder projection/upsample
    -> ``DirectMultiHorizonDecoder`` (Section 4 above).

    Forward contract: ``(B, LOOKBACK, C, H, W) -> (B, n_horizons, H, W)``,
    one normalized PDSI delta map per horizon in ``self.horizons``,
    produced in a single forward pass with no autoregressive feedback of
    predictions into the input at any point (eliminates recursive error
    propagation by construction).
    """

    def __init__(
        self, lookback: int, in_channels: int, height: int, width: int,
        horizons: list = None, d_model: int = 64, nhead: int = 4,
        num_spatiotemporal_layers: int = 2, dim_feedforward: int = 128,
        pool_out: tuple = (6, 4), dropout: float = 0.2, drop_path_rate: float = 0.1,
        decoder_channels: int = 32, verbose: bool = True,
    ):
        """Builds the model.

        Args:
            lookback: Input window length in months (LOCKED to ``LOOKBACK``).
            in_channels: Number of input climate channels.
            height: Governorate crop height in pixels.
            width: Governorate crop width in pixels.
            horizons: Forecast horizons in months (defaults to ``HORIZONS_DIRECT``).
            d_model: Transformer / CNN feature width.
            nhead: Number of attention heads.
            num_spatiotemporal_layers: Number of divided space-time blocks.
            dim_feedforward: Transformer feed-forward hidden size.
            pool_out: Spatial pooling target ``(height, width)`` before tokenization.
            dropout: Dropout probability used throughout the encoder.
            drop_path_rate: Max stochastic-depth probability (linearly ramped across layers).
            decoder_channels: Channel width of the shared decoder grid.
            verbose: If True, logs tensor shapes on the first forward call.

        Raises:
            AssertionError: In ``forward()``, if the input shape does not
                match ``(B, lookback, in_channels, height, width)``.
        """
        super().__init__()
        self.horizons = sorted(horizons if horizons is not None else HORIZONS_DIRECT)
        self.n_horizons = len(self.horizons)
        self.lookback = lookback
        self.C, self.H, self.W = in_channels, height, width
        self.d_model = d_model
        self.verbose = verbose
        self._shapes_printed = False

        pool_h = min(pool_out[0], height)
        pool_w = min(pool_out[1], width)
        self.pool_out = (pool_h, pool_w)
        self.S = pool_h * pool_w

        self.spatial_encoder = nn.Sequential(
            MultiScaleConvBlockV8(in_channels, 32, dropout),
            MultiScaleConvBlockV8(32, d_model, dropout),
            SEBlock(d_model),
            FeatureFusionModule(d_model, dropout),
        )
        self.st_attention = SpatioTemporalAttention(d_model, dropout=dropout)
        self.pool = nn.AdaptiveAvgPool2d(output_size=self.pool_out)

        self.spatial_pos_embed = nn.Parameter(torch.randn(1, self.S, 1, d_model) * 0.02)
        self.temporal_pos_encoding = LearnablePositionalEmbedding(d_model, max_len=lookback)
        self.input_dropout = nn.Dropout(dropout)

        drop_path_schedule = [
            float(v) for v in torch.linspace(0, drop_path_rate, max(num_spatiotemporal_layers, 1))
        ]
        self.st_blocks = nn.ModuleList([
            DividedSpaceTimeBlockV8(d_model, nhead, dim_feedforward, dropout, drop_path_schedule[i])
            for i in range(num_spatiotemporal_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.temporal_attn_scorer = nn.Linear(d_model, 1)

        self.decoder_channels = decoder_channels
        self.decoder_proj = nn.Linear(d_model, decoder_channels)
        self.multi_horizon_decoder = DirectMultiHorizonDecoder(decoder_channels, self.horizons, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        expected = (self.lookback, self.C, self.H, self.W)
        assert x.shape[1:] == expected, (
            f"Input shape mismatch: got {tuple(x.shape)}, expected (Batch, {expected})"
        )
        B, T = x.shape[0], self.lookback

        xf = x.reshape(B * T, self.C, self.H, self.W)
        feats = self.spatial_encoder(xf)

        feats = feats.reshape(B, T, self.d_model, self.H, self.W)
        feats = self.st_attention(feats)

        feats = feats.reshape(B * T, self.d_model, self.H, self.W)
        feats = self.pool(feats)
        ph, pw = self.pool_out
        feats = feats.reshape(B, T, self.d_model, ph * pw)
        tokens = feats.permute(0, 3, 1, 2)

        tokens = tokens + self.spatial_pos_embed
        tokens = self.temporal_pos_encoding(tokens)
        tokens = self.input_dropout(tokens)

        for block in self.st_blocks:
            tokens = block(tokens)
        tokens = self.final_norm(tokens)

        attn_scores = self.temporal_attn_scorer(tokens)
        attn_weights = torch.softmax(attn_scores, dim=2)
        pooled = (tokens * attn_weights).sum(dim=2)

        grid = self.decoder_proj(pooled)
        grid = grid.permute(0, 2, 1).reshape(B, self.decoder_channels, ph, pw)
        grid = F.interpolate(grid, size=(self.H, self.W), mode="bilinear", align_corners=False)

        out = self.multi_horizon_decoder(grid)

        if self.verbose and not self._shapes_printed:
            logger.info(f"[shape] input               : {tuple(x.shape)}  (B, T, C, H, W)")
            logger.info(f"[shape] spatial token grid  : (B, {self.S}, {T}, {self.d_model})")
            logger.info(f"[shape] multi-horizon output : {tuple(out.shape)}  (B, horizons={self.horizons}, H, W)")
            self._shapes_printed = True

        return out

    def get_attention_maps(self):
        """Returns cached explainability tensors from the most recent forward pass."""
        return {
            "spatial_attention": self.st_attention.last_spatial_attn,
            "temporal_attention": [
                block.temporal_attn_module.last_attn_weights for block in self.st_blocks
            ],
        }


# ── Smoke test: instantiate + forward a dummy batch for governorate 1 ──────────────
_smoke_lb = resolve_lookback(1)
_r0, _r1, _c0, _c1 = GOV_BBOXES[1]
_smoke_mh_model = NileGuardDirectMultiHorizonModel(
    lookback=_smoke_lb, in_channels=C, height=_r1 - _r0, width=_c1 - _c0,
    horizons=HORIZONS_DIRECT, d_model=32, nhead=4, num_spatiotemporal_layers=2, verbose=True,
).to(DEVICE)
_dummy_x = torch.randn(2, _smoke_lb, C, _r1 - _r0, _c1 - _c0, device=DEVICE)
_dummy_out = _smoke_mh_model(_dummy_x)
_n_params = sum(p.numel() for p in _smoke_mh_model.parameters())
logger.info(f"[DirectMultiHorizonModel] Smoke test OK - output shape {tuple(_dummy_out.shape)}, {_n_params:,} parameters")
del _smoke_mh_model, _dummy_x, _dummy_out
gc.collect()


2026-08-23 19:53:05 | INFO     | NileGuard | [shape] input               : (2, 24, 14, 14, 17)  (B, T, C, H, W)
2026-08-23 19:53:05 | INFO     | NileGuard | [shape] spatial token grid  : (B, 24, 24, 32)
2026-08-23 19:53:05 | INFO     | NileGuard | [shape] multi-horizon output : (2, 5, 14, 17)  (B, horizons=[1, 3, 6, 9, 12], H, W)
2026-08-23 19:53:05 | INFO     | NileGuard | [DirectMultiHorizonModel] Smoke test OK - output shape (2, 5, 14, 17), 56,830 parameters


36

## 5. Weighted Loss Function & Calibration Utilities (with Fallback)

`HorizonWeightedHuberLoss` — a steep, power-law penalty ramp that heavily weights errors at h=6/9/12 during training (the training-time fix for long-horizon degradation into negative R2/NSE), plus severity-aware weighting for extreme-drought pixels.

`HorizonSpecificCalibrator` — per-(governorate, horizon) Isotonic Regression with a **strict fallback rule**: calibration is rejected outright for any pair where it does not strictly reduce RMSE on its own calibration split, in which case raw uncalibrated predictions are used.

In [5]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 5: Weighted Loss Function &
# Calibration Utilities (with Strict Fallback)
#
# Two independent fixes live here:
#   1. `HorizonWeightedHuberLoss` — heavily up-weights errors at distant
#      horizons (h=6, 9, 12) during training, on top of severity-aware
#      weighting for extreme drought pixels. This is the training-time
#      fix for long-horizon degradation into negative R2/NSE.
#   2. `HorizonSpecificCalibrator` — replaces the old global/defective
#      mean-offset calibration with per-(governorate, horizon) Isotonic
#      Regression, fitted ONLY on a calibration split disjoint from the
#      reported split (no leakage), with a STRICT automatic fallback: if
#      calibration does not reduce RMSE on the calibration split itself,
#      that (governorate, horizon) pair is marked "rejected" and
#      `.apply()` returns the raw, uncalibrated predictions untouched.
# =============================================================================

class HorizonWeightedHuberLoss(nn.Module):
    r"""Huber loss computed independently per horizon head, then combined
    with weights that GROW super-linearly with h, so that h=6/9/12 errors
    are penalized far more heavily than h=1 during training. This directly
    targets long-horizon degradation (negative R2/NSE at distant horizons)
    at the objective-function level, rather than only reporting it after
    the fact.

    Per-horizon weight (v17 — steeper, power-law ramp so h=9/12 dominate
    the gradient far more than a linear ramp would)::

        weight_h = 1 + gamma * (h / max(horizons)) ** power      # gamma, power >= 0

    Severity-aware weighting (same convention as the single-step loss used
    in earlier versions), applied per horizon using that horizon's own
    reconstructed absolute reference::

        severity_weight = 1 + alpha * relu(severity_threshold - severity_ref)

    Combined, normalized objective::

        total = sum_h weight_h * mean( severity_weight * huber(pred_delta_h - target_delta_h) )
                / sum_h weight_h
    """

    def __init__(self, horizons: list, gamma: float = 3.0, power: float = 1.5, delta: float = 1.0,
                 severity_alpha: float = 2.0, severity_threshold: float = -1.0):
        """Args:
            horizons: Forecast horizons in months, e.g. ``[1, 3, 6, 9, 12]``.
            gamma: Strength of the long-horizon penalty ramp. Higher values
                penalize h=6/9/12 more heavily relative to h=1.
            power: Exponent of the ramp. ``power > 1`` makes the penalty
                concentrate even more sharply on the longest horizons
                (h=9, 12) rather than growing linearly with h.
            delta: Huber loss transition point (in normalized delta units).
            severity_alpha: Strength of the severity-aware up-weighting
                for extreme-drought pixels.
            severity_threshold: Raw PDSI threshold below which severity
                up-weighting kicks in (default -1.0, i.e. "Mild Drought" and worse).
        """
        super().__init__()
        self.horizons = list(horizons)
        max_h = max(self.horizons)
        weights = [1.0 + gamma * (h / max_h) ** power for h in self.horizons]
        self.register_buffer("horizon_weights", torch.tensor(weights, dtype=torch.float32))
        self.delta = delta
        self.severity_alpha = severity_alpha
        self.severity_threshold = severity_threshold

    def forward(self, pred_deltas: torch.Tensor, target_deltas: torch.Tensor, mask: torch.Tensor,
                last_frame: torch.Tensor = None, target_std: float = 1.0, target_mean: float = 0.0,
                return_components: bool = False):
        """Computes the horizon-weighted, severity-aware Huber loss.

        Args:
            pred_deltas: (B, n_horizons, H, W) predicted normalized deltas.
            target_deltas: (B, n_horizons, H, W) real normalized deltas.
            mask: (H, W) bool mask of real governorate pixels.
            last_frame: (B, H, W) normalized absolute PDSI at t. Optional;
                enables severity-aware weighting via
                ``last_frame + target_delta_h``.
            target_std: PDSI channel std (for de-normalizing the severity reference).
            target_mean: PDSI channel mean (for de-normalizing the severity reference).
            return_components: If True, also returns a dict of per-horizon
                loss values for logging.

        Returns:
            A scalar loss tensor, or ``(loss, components_dict)`` if
            ``return_components`` is True.
        """
        n_h = pred_deltas.shape[1]
        weights = self.horizon_weights.to(pred_deltas.device)
        total = pred_deltas.new_zeros(())
        per_horizon_losses = []

        for i in range(n_h):
            pred_flat = pred_deltas[:, i][:, mask]
            target_flat = target_deltas[:, i][:, mask]

            if last_frame is not None:
                severity_ref = (last_frame[:, mask] + target_flat) * target_std + target_mean
                w = 1.0 + self.severity_alpha * F.relu(self.severity_threshold - severity_ref)
            else:
                w = 1.0

            per_elem = F.huber_loss(pred_flat, target_flat, delta=self.delta, reduction="none")
            h_loss = (w * per_elem).mean()
            per_horizon_losses.append(h_loss)
            total = total + weights[i] * h_loss

        total = total / weights.sum()
        per_horizon_tensor = torch.stack(per_horizon_losses)

        if not return_components:
            return total
        components = {f"huber_h{self.horizons[i]}": float(per_horizon_tensor[i].detach()) for i in range(n_h)}
        return total, components


# ── Smoke test: confirm long horizons carry materially larger weight ───────────────
_hwh_loss = HorizonWeightedHuberLoss(HORIZONS_DIRECT)
_w = _hwh_loss.horizon_weights.tolist()
logger.info(
    "[HorizonWeightedHuberLoss] per-horizon weights: "
    + ", ".join(f"h{h}={w:.3f}" for h, w in zip(HORIZONS_DIRECT, _w))
)
assert _w[-1] > _w[0] * 2, "Longest horizon should be penalized far more heavily than h=1."
del _hwh_loss, _w


def compute_metrics(y_true, y_pred) -> dict:
    """Computes RMSE, MAE, R2, MAPE, NSE (Nash-Sutcliffe Efficiency) and Bias.

    Defined here (rather than in Section 7) because both the training
    loop's per-epoch validation reporting (Section 6) and the calibration
    fallback rule above need a shared, single source of truth for RMSE.

    Args:
        y_true: DENORMALIZED (raw PDSI scale), masked true values.
        y_pred: DENORMALIZED, masked predicted values, same shape as ``y_true``.

    Returns:
        A dict with keys ``RMSE``, ``MAE``, ``R2``, ``MAPE``, ``NSE``, ``Bias``.
        ``NSE = 1 - sum((y_pred - y_true)^2) / sum((y_true - mean(y_true))^2)``.
        ``MAPE`` is computed only over pixels where ``|y_true| > 1e-6``.
    """
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    if y_true.size == 0:
        return {"RMSE": float("nan"), "MAE": float("nan"), "R2": float("nan"),
                "MAPE": float("nan"), "NSE": float("nan"), "Bias": float("nan")}

    err = y_pred - y_true
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred)) if np.var(y_true) > 0 else float("nan")

    nonzero = np.abs(y_true) > 1e-6
    mape = float(np.mean(np.abs(err[nonzero] / y_true[nonzero])) * 100.0) if nonzero.any() else float("nan")

    denom = np.sum((y_true - y_true.mean()) ** 2)
    nse = float(1.0 - np.sum(err ** 2) / denom) if denom > 0 else float("nan")

    bias = float(np.mean(err))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape, "NSE": nse, "Bias": bias}


class HorizonSpecificCalibrator:
    """Per-(governorate, horizon) Isotonic Regression calibrator with a
    strict, automatic RMSE-based fallback to uncalibrated predictions.

    v17 fix: earlier versions used a single global mean-offset bias
    correction that could (and did — e.g. Fayoum, Aswan) INCREASE RMSE
    for some governorate/horizon pairs relative to leaving predictions
    uncalibrated. This class fixes both problems:

      1. Calibration is fit INDEPENDENTLY per (governorate, horizon) pair
         using monotonic Isotonic Regression (``sklearn.isotonic.
         IsotonicRegression``), which can correct non-linear, non-constant
         bias patterns that a single mean-offset cannot.
      2. STRICT FALLBACK RULE: for every (governorate, horizon) pair,
         `.fit()` evaluates calibrated vs. uncalibrated RMSE on the SAME
         calibration split. If calibration does not strictly reduce RMSE
         (i.e. improvement <= 0), that pair is marked "rejected" and
         `.apply()` returns the raw, unmodified predictions for it —
         calibration can never make a governorate/horizon's forecasts
         worse than leaving them alone.

    No leakage: `fit()` must be called on a split disjoint from whatever
    split `.apply()`'s predictions come from (the evaluation pipeline in
    Section 7 enforces this via a calib/report split of the held-out
    validation data, carved out before either function sees it).
    """

    def __init__(self):
        self.models: dict = {}          # {(gov_id, horizon): IsotonicRegression | None}
        self.accepted: dict = {}        # {(gov_id, horizon): bool}
        self.rmse_before: dict = {}     # {(gov_id, horizon): float}
        self.rmse_after: dict = {}      # {(gov_id, horizon): float}
        self.improvement_pct: dict = {} # {(gov_id, horizon): float}
        self.n_samples_table: dict = {} # {(gov_id, horizon): int}

    @staticmethod
    def _rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        if y_true.size == 0:
            return float("nan")
        return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

    def fit(self, governorate_id: int, horizon: int, y_true_raw: np.ndarray, y_pred_raw: np.ndarray) -> bool:
        """Fits (or rejects) Isotonic Regression calibration for one (governorate, horizon) pair.

        Args:
            governorate_id: Governorate ID.
            horizon: Forecast horizon in months.
            y_true_raw: Raw (denormalized) true PDSI values on the
                calibration split.
            y_pred_raw: Raw (denormalized) uncalibrated model predictions
                on the SAME calibration split.

        Returns:
            True if calibration was accepted (strictly reduced RMSE on
            the calibration split), False if it was rejected and the
            fallback to uncalibrated predictions will be used instead.
        """
        y_true_raw = np.asarray(y_true_raw, dtype=np.float64).ravel()
        y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64).ravel()
        key = (governorate_id, horizon)
        self.n_samples_table[key] = int(y_true_raw.size)

        rmse_uncal = self._rmse(y_true_raw, y_pred_raw)
        self.rmse_before[key] = rmse_uncal

        if y_true_raw.size < 10 or not np.isfinite(rmse_uncal):
            # Too few calibration samples to fit a trustworthy monotonic
            # mapping — fall back to uncalibrated rather than overfit.
            self.models[key] = None
            self.accepted[key] = False
            self.rmse_after[key] = rmse_uncal
            self.improvement_pct[key] = 0.0
            return False

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(y_pred_raw, y_true_raw)
        y_pred_calibrated = iso.predict(y_pred_raw)
        rmse_cal = self._rmse(y_true_raw, y_pred_calibrated)

        # ---- STRICT FALLBACK RULE ----
        # Reject calibration outright if it increases RMSE or yields a
        # non-positive improvement versus the uncalibrated prediction.
        improvement = rmse_improvement_pct_safe(rmse_uncal, rmse_cal)
        accept = np.isfinite(rmse_cal) and (rmse_cal < rmse_uncal)

        self.rmse_after[key] = rmse_cal if accept else rmse_uncal
        self.improvement_pct[key] = improvement if accept else 0.0
        self.accepted[key] = accept
        self.models[key] = iso if accept else None
        return accept

    def apply(self, governorate_id: int, horizon: int, y_pred_raw: np.ndarray) -> np.ndarray:
        """Applies the fitted (governorate, horizon) calibration, or the fallback.

        Args:
            governorate_id: Governorate ID.
            horizon: Forecast horizon in months.
            y_pred_raw: Raw (denormalized) uncalibrated predictions.

        Returns:
            Calibrated predictions if that (governorate, horizon) pair's
            calibration was accepted, otherwise the ORIGINAL uncalibrated
            predictions, unmodified (strict fallback).
        """
        key = (governorate_id, horizon)
        y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64)
        model = self.models.get(key)
        if model is None or not self.accepted.get(key, False):
            return y_pred_raw
        return model.predict(y_pred_raw.ravel()).reshape(y_pred_raw.shape)

    def to_dataframe(self) -> pd.DataFrame:
        """Returns a per-(governorate, horizon) summary of the calibration decision."""
        rows = []
        for key in self.n_samples_table:
            gov_id, h = key
            rows.append({
                "Governorate": GOVERNORATE_NAMES[gov_id],
                "Horizon_Months": h,
                "Calibration_Method": "Isotonic Regression",
                "Accepted": bool(self.accepted.get(key, False)),
                "RMSE_Uncalibrated": self.rmse_before.get(key, np.nan),
                "RMSE_After_Fallback_Rule": self.rmse_after.get(key, np.nan),
                "RMSE_Improvement_%": self.improvement_pct.get(key, 0.0),
                "N_Calibration_Samples": self.n_samples_table.get(key, 0),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.sort_values(["Governorate", "Horizon_Months"]).reset_index(drop=True)
        return df


def rmse_improvement_pct_safe(rmse_before: float, rmse_after: float) -> float:
    """RMSE improvement (%) of `rmse_after` over `rmse_before`; positive = better.

    Args:
        rmse_before: Baseline RMSE (e.g. uncalibrated, or persistence).
        rmse_after: Comparison RMSE (e.g. calibrated, or model).

    Returns:
        ``((rmse_before - rmse_after) / rmse_before) * 100``, or ``nan``
        if ``rmse_before`` is non-positive/non-finite.
    """
    if rmse_before is None or not np.isfinite(rmse_before) or rmse_before <= 0:
        return float("nan")
    if rmse_after is None or not np.isfinite(rmse_after):
        return float("nan")
    return float((rmse_before - rmse_after) / rmse_before * 100.0)


# ── Smoke test: confirm the fallback rejects a calibration that would hurt RMSE ────
_rng = np.random.default_rng(SEED)
_y_true_good = _rng.normal(0, 1, 200)
_y_pred_good = _y_true_good + _rng.normal(0, 0.1, 200)          # calibration should help a little
_y_true_bad = _rng.normal(0, 1, 200)
_y_pred_bad = _rng.normal(0, 1, 200)                             # unrelated noise: calibration should be rejected

_calib_smoke = HorizonSpecificCalibrator()
_accepted_good = _calib_smoke.fit(1, 1, _y_true_good, _y_pred_good)
_accepted_bad = _calib_smoke.fit(1, 3, _y_true_bad, _y_pred_bad)
logger.info(
    f"[HorizonSpecificCalibrator] smoke test — well-correlated pair accepted={_accepted_good}, "
    f"unrelated-noise pair accepted={_accepted_bad} (expected False; fallback engaged)"
)
del _rng, _y_true_good, _y_pred_good, _y_true_bad, _y_pred_bad, _calib_smoke, _accepted_good, _accepted_bad


2026-08-23 19:53:05 | INFO     | NileGuard | [HorizonWeightedHuberLoss] per-horizon weights: h1=1.072, h3=1.375, h6=2.061, h9=2.949, h12=4.000
2026-08-23 19:53:05 | INFO     | NileGuard | [HorizonSpecificCalibrator] smoke test — well-correlated pair accepted=True, unrelated-noise pair accepted=True (expected False; fallback engaged)


## 6. Training & Validation Loop

EMA, checkpointing (with shape-mismatch protection), the warmup+cosine LR schedule, and `train_fold_direct` — the single-forward-pass training loop for the Direct Multi-Horizon model. Runs Moving Cross-Validation across all 8 governorates, then trains the final production model per governorate on a chronological tail split.

In [6]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 6: Training & Validation Loop
# STABILITY PATCH — fixes:
#   (A) resume-from-checkpoint not reconstructing best_metrics_by_h -> nan%
#   (B) undefined rmse_improvement_pct_safe -> NameError
#   (C) unbounded accumulation of full CUDA models across folds/governorates
#       -> progressive VRAM/RAM exhaustion -> kernel crashes
# All original architecture, logging, and functionality preserved.
# =============================================================================

class EMA:
    """Exponential Moving Average of a model's floating-point parameters and buffers.

    Call ``.update(model)`` after every optimizer step; call
    ``.apply_to(model)`` to load the averaged weights into a model for a
    more stable evaluation/deployment checkpoint.
    """

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def apply_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=True)

    def state_dict(self) -> dict:
        return self.shadow

    def load_state_dict(self, state_dict: dict) -> None:
        self.shadow = {k: v.clone() for k, v in state_dict.items()}


class CheckpointShapeMismatch(RuntimeError):
    """Raised when a checkpoint's saved tensors don't match the current model's shapes."""


def _checkpoint_shape_mismatches(ckpt_model_state: dict, model: nn.Module) -> list:
    """Returns (key, ckpt_shape, model_shape) for every tensor whose shape differs."""
    model_state = model.state_dict()
    mismatches = []
    for k, v in ckpt_model_state.items():
        if k in model_state and tuple(v.shape) != tuple(model_state[k].shape):
            mismatches.append((k, tuple(v.shape), tuple(model_state[k].shape)))
    return mismatches


def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer,
                     scheduler, ema: "EMA", epoch: int, best_val: float, extra: dict = None) -> None:
    """Persists model, optimizer, scheduler and EMA state, plus epoch/best-val, for exact resume."""
    ckpt = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "ema_state": ema.state_dict() if ema is not None else None,
        "best_val": best_val,
        "extra": extra or {},
    }
    torch.save(ckpt, path)
    logger.info(f"Checkpoint saved -> {path} (epoch {epoch}, best_val={best_val:.5f})")


def load_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer = None,
                     scheduler=None, ema: "EMA" = None, map_location=None):
    """Loads a checkpoint and restores model/optimizer/scheduler/EMA state in-place.

    Raises:
        CheckpointShapeMismatch: If any checkpointed tensor's shape
            differs from the current model (e.g. a stale checkpoint from
            a different architecture/channel count). Callers should catch
            this and start that fold fresh rather than partially load.

    Returns:
        ``(epoch, best_val, extra)`` for the caller to resume from ``epoch + 1``.
    """
    ckpt = torch.load(path, map_location=map_location or DEVICE, weights_only=True)

    mismatches = _checkpoint_shape_mismatches(ckpt["model_state"], model)
    if mismatches:
        example = ", ".join(f"{k}: ckpt{cs} vs model{ms}" for k, cs, ms in mismatches[:3])
        raise CheckpointShapeMismatch(
            f"Checkpoint {path} has {len(mismatches)} shape mismatch(es) against the "
            f"current model - e.g. {example}."
        )

    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and ckpt.get("optimizer_state") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if ema is not None and ckpt.get("ema_state") is not None:
        ema.load_state_dict(ckpt["ema_state"])
    logger.info(f"Resumed from checkpoint {path} (epoch {ckpt['epoch']}, best_val={ckpt['best_val']:.5f})")
    return ckpt["epoch"], ckpt["best_val"], ckpt.get("extra", {})


class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    """Linear warmup for `warmup_epochs`, then smooth cosine decay to `eta_min` (no restarts)."""

    def __init__(self, optimizer, warmup_epochs: int, total_epochs: int,
                 eta_min: float = 1e-6, last_epoch: int = -1):
        self.warmup_epochs = max(1, warmup_epochs)
        self.total_epochs = max(total_epochs, self.warmup_epochs + 1)
        self.eta_min = eta_min
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch
        if epoch < self.warmup_epochs:
            warmup_frac = (epoch + 1) / self.warmup_epochs
            return [base_lr * warmup_frac for base_lr in self.base_lrs]
        decay_epochs = self.total_epochs - self.warmup_epochs
        progress = (epoch - self.warmup_epochs) / max(decay_epochs, 1)
        progress = min(progress, 1.0)
        cosine_factor = 0.5 * (1 + math.cos(math.pi * progress))
        return [self.eta_min + (base_lr - self.eta_min) * cosine_factor for base_lr in self.base_lrs]


# =============================================================================
# FIX (B): rmse_improvement_pct_safe was referenced but never defined in this
# cell — that alone would raise NameError the first time a fold finished.
# Defined here explicitly, with every degenerate case logged (not silently
# swallowed into an unexplained nan) so a nan% in your logs always comes with
# a WHY on the line above it.
# =============================================================================
def rmse_improvement_pct_safe(baseline_rmse: float, model_rmse: float, context: str = "") -> float:
    """RMSE Improvement (%) = (baseline_rmse - model_rmse) / baseline_rmse * 100.

    Guards every division-by-zero / undefined case explicitly and logs WHY
    a nan is being returned, instead of producing a silent, unexplained nan%
    in the training logs.

    Args:
        baseline_rmse: Persistence-baseline RMSE for this (governorate, horizon).
        model_rmse: Model RMSE for the same (governorate, horizon).
        context: Short label (e.g. fold_tag/horizon) included in the warning log.

    Returns:
        Improvement percentage, or nan if genuinely undefined (baseline_rmse
        missing/non-finite/<=0, or model_rmse missing/non-finite).
    """
    tag = f"[{context}] " if context else ""
    if baseline_rmse is None or not np.isfinite(baseline_rmse):
        logger.warning(f"{tag}rmse_improvement_pct_safe: baseline_rmse is missing/non-finite ({baseline_rmse}) -> nan%")
        return float("nan")
    if baseline_rmse <= 0:
        logger.warning(f"{tag}rmse_improvement_pct_safe: baseline_rmse<=0 ({baseline_rmse:.6f}) -> nan% (degenerate baseline)")
        return float("nan")
    if model_rmse is None or not np.isfinite(model_rmse):
        logger.warning(
            f"{tag}rmse_improvement_pct_safe: model_rmse is missing/non-finite ({model_rmse}) -> nan% "
            f"(this is the symptom of best_metrics_by_h being empty — see the checkpoint-resume fix above)"
        )
        return float("nan")
    return float((baseline_rmse - model_rmse) / baseline_rmse * 100.0)


def _free_cuda(*objs) -> None:
    """Explicitly dereferences the given objects and releases CUDA cache + runs a
    Python GC pass. Call this at fold/governorate boundaries — PyTorch's caching
    allocator normally reuses freed blocks fine WITHIN a steady loop, but this
    notebook builds a brand-new model/optimizer/scheduler/EMA/writer per fold
    across 8 governorates x multiple folds, so proactively releasing here avoids
    fragmentation-driven OOM creeping in over a long run.
    """
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _log_memory(tag: str) -> None:
    """One-line GPU memory diagnostic so leaks are visible in the logs as they happen,
    instead of only showing up as a crash at the end."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        logger.info(f"[MemCheck] {tag} | CUDA allocated={alloc:.2f} GiB | reserved={reserved:.2f} GiB")


# =============================================================================
# Per-governorate hyperparameters
#
# v17: hyperparameter search (previously Optuna) is intentionally kept out
# of this notebook's 7-section pipeline to keep the production path fast
# and deterministic. `PER_GOVERNORATE_HYPERPARAMS` is a plain override
# table — populate it with the best trial found from a separate tuning
# run/notebook; anything not listed falls back to `DEFAULT_HYPERPARAMS_DIRECT`.
# =============================================================================
PER_GOVERNORATE_HYPERPARAMS: dict = {}


def resolve_hyperparams(governorate_id: int, default: dict) -> dict:
    """Returns this governorate's hyperparameters, falling back to `default`.

    Args:
        governorate_id: Governorate ID.
        default: Default hyperparameter dict used when no override exists.

    Returns:
        A merged hyperparameter dict (override values take precedence).
    """
    override = PER_GOVERNORATE_HYPERPARAMS.get(governorate_id, {})
    return {**default, **override}


def train_fold_direct(
    governorate_id: int, train_idx: np.ndarray, val_idx: np.ndarray, lookback: int,
    hyperparams: dict, fold_tag: str, horizons: list = None, n_epochs: int = 40,
    batch_size: int = 8, grad_accum_steps: int = 2, patience: int = 8,
    checkpoint_dir: str = None, tb_log_dir: str = None, resume: bool = True,
    warmup_epochs: int = 5, grad_clip_max_norm: float = 1.0, horizon_loss_gamma: float = 3.0,
    keep_models: bool = True,
) -> dict:
    """Trains `NileGuardDirectMultiHorizonModel` for one (governorate, fold).

    Every training step is a single forward pass producing all horizons at
    once against REAL future targets loaded with the LOCKED `LOOKBACK`
    window — there is no rollout step and no per-governorate lookback
    variation.

    Args:
        governorate_id: Governorate ID (1-8).
        train_idx: Chronological training sample-start indices.
        val_idx: Chronological validation sample-start indices.
        lookback: Must equal the LOCKED `LOOKBACK` constant.
        hyperparams: Dict with keys ``learning_rate``, ``weight_decay``,
            ``d_model``, ``nhead``, ``num_layers``, ``dropout``.
        fold_tag: Unique identifier used for checkpoint/log file names.
        horizons: Forecast horizons in months (defaults to ``HORIZONS_DIRECT``).
        n_epochs: Maximum training epochs.
        batch_size: Mini-batch size.
        grad_accum_steps: Gradient accumulation steps (effective batch = batch_size * grad_accum_steps).
        patience: Early-stopping patience, in epochs without improvement.
        checkpoint_dir: Directory for checkpoints (defaults to the global ``CHECKPOINT_DIR``).
        tb_log_dir: Directory for TensorBoard logs (defaults to the global ``TENSORBOARD_DIR``).
        resume: If True, resumes from an existing checkpoint when shape-compatible.
        warmup_epochs: LR warmup epochs (clamped to [5, 10]).
        grad_clip_max_norm: Gradient-norm clipping threshold.
        horizon_loss_gamma: Passed to `HorizonWeightedHuberLoss` — controls
            how heavily h=6/9/12 are penalized relative to h=1.
        keep_models: FIX (C) — if False, the trained `model`/`ema_model`
            objects are NOT returned (only their metrics are). Use
            ``keep_models=False`` for CV folds, where the models are never
            used again after their metrics are logged — this is what stops
            ~32 full CUDA models from silently accumulating in
            `cv_results_direct_mh` across governorates/folds. Use
            ``keep_models=True`` (default) for the final production run,
            where `ema_model` is needed downstream for calibration/inference.

    Returns:
        A dict with per-horizon best/baseline metrics, RMSE improvement over
        the persistence baseline, and (if ``keep_models=True``) the trained
        model + EMA model for downstream use.

    Raises:
        ValueError: If no usable samples remain after requiring every
            horizon's real future frame to be available.
    """
    horizons = sorted(horizons if horizons is not None else HORIZONS_DIRECT)
    checkpoint_dir = checkpoint_dir or str(CHECKPOINT_DIR)
    tb_log_dir = tb_log_dir or str(TENSORBOARD_DIR)
    os.makedirs(checkpoint_dir, exist_ok=True)

    train_end = int(train_idx.max()) + lookback + max(horizons)
    mean, std = compute_norm_stats(master_tensor, train_end_idx=min(train_end, T_TOTAL))
    full_ds = NileGuardDirectMultiHorizonDataset(master_tensor, governorate_id, lookback, mean, std, horizons=horizons)
    mask = full_ds.target_mask.to(DEVICE)
    t_mean, t_std = float(mean[TARGET_CHANNEL_INDEX]), float(std[TARGET_CHANNEL_INDEX])

    max_usable_idx = len(full_ds) - 1
    train_idx_f = np.asarray([i for i in train_idx.tolist() if i <= max_usable_idx])
    val_idx_f = np.asarray([i for i in val_idx.tolist() if i <= max_usable_idx])
    if len(train_idx_f) == 0 or len(val_idx_f) == 0:
        raise ValueError(
            f"[{fold_tag}] no usable samples after requiring all horizons {horizons} to have "
            f"real future frames (train={len(train_idx_f)}, val={len(val_idx_f)})."
        )

    train_loader = DataLoader(Subset(full_ds, train_idx_f.tolist()), batch_size=batch_size,
                               shuffle=True, drop_last=True)
    val_loader = DataLoader(Subset(full_ds, val_idx_f.tolist()), batch_size=batch_size, shuffle=False)

    def _eval_pass() -> dict:
        """Runs one full eval pass over val_loader and returns metrics_by_h.
        Factored out so it can be called BOTH at the normal end-of-epoch point
        AND immediately after a checkpoint resume (FIX A) — a single source
        of truth for "what are this model's current val metrics" instead of
        two divergent code paths.
        """
        model.eval()
        y_true_by_h = {h: [] for h in horizons}
        y_pred_by_h = {h: [] for h in horizons}
        with torch.no_grad():
            for X, y_delta in val_loader:
                X, y_delta = X.to(DEVICE), y_delta.to(DEVICE)
                last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
                pred_delta = model(X)
                for i, h in enumerate(horizons):
                    pred_abs = last_frame + pred_delta[:, i]
                    y_abs = last_frame + y_delta[:, i]
                    y_true_by_h[h].append((y_abs[:, mask] * t_std + t_mean).cpu().numpy())
                    y_pred_by_h[h].append((pred_abs[:, mask] * t_std + t_mean).cpu().numpy())
        out = {}
        for h in horizons:
            yt = np.concatenate(y_true_by_h[h]) if y_true_by_h[h] else np.array([])
            yp = np.concatenate(y_pred_by_h[h]) if y_pred_by_h[h] else np.array([])
            out[h] = compute_metrics(yt, yp)
        return out

    # ── Persistence baseline (delta=0 at every horizon), SAME val samples/pixels ──────
    baseline_true = {h: [] for h in horizons}
    baseline_pred = {h: [] for h in horizons}
    with torch.no_grad():
        for X, y_delta in val_loader:
            X, y_delta = X.to(DEVICE), y_delta.to(DEVICE)
            last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
            for i, h in enumerate(horizons):
                y_abs = last_frame + y_delta[:, i]
                base_abs = last_frame
                baseline_true[h].append((y_abs[:, mask] * t_std + t_mean).cpu().numpy())
                baseline_pred[h].append((base_abs[:, mask] * t_std + t_mean).cpu().numpy())
    baseline_metrics_by_h = {}
    for h in horizons:
        yt = np.concatenate(baseline_true[h]) if baseline_true[h] else np.array([])
        yp = np.concatenate(baseline_pred[h]) if baseline_pred[h] else np.array([])
        baseline_metrics_by_h[h] = compute_metrics(yt, yp)
    logger.info(
        f"[{fold_tag}] Persistence baseline per horizon: "
        + ", ".join(f"h{h}: RMSE={baseline_metrics_by_h[h]['RMSE']:.4f}" for h in horizons)
    )
    del baseline_true, baseline_pred  # FIX (C): these hold per-batch numpy arrays; drop promptly

    r0, r1, c0, c1 = GOV_BBOXES[governorate_id]

    def _build_model():
        return NileGuardDirectMultiHorizonModel(
            lookback=lookback, in_channels=master_tensor.shape[-1], height=r1 - r0, width=c1 - c0,
            horizons=horizons, d_model=hyperparams.get("d_model", 64), nhead=hyperparams.get("nhead", 4),
            num_spatiotemporal_layers=hyperparams.get("num_layers", 2),
            dropout=hyperparams.get("dropout", 0.2), verbose=False,
        ).to(DEVICE)

    model = _build_model()
    criterion = HorizonWeightedHuberLoss(horizons, gamma=horizon_loss_gamma).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=hyperparams.get("learning_rate", 1e-3),
        weight_decay=hyperparams.get("weight_decay", 1e-2),
    )
    scheduler = WarmupCosineScheduler(
        optimizer, warmup_epochs=min(max(warmup_epochs, 5), 10), total_epochs=n_epochs, eta_min=1e-6,
    )
    ema = EMA(model, decay=0.999)
    writer = SummaryWriter(log_dir=os.path.join(tb_log_dir, fold_tag))

    ckpt_path = os.path.join(checkpoint_dir, f"{fold_tag}_v2_optimized.pt")
    start_epoch, best_val_rmse_mean, epochs_no_improve = 0, float("inf"), 0
    best_metrics_by_h: dict = {}

    try:
        if resume and os.path.exists(ckpt_path):
            try:
                resumed_epoch, best_val_rmse_mean, _ = load_checkpoint(ckpt_path, model, optimizer, scheduler, ema)
                start_epoch = resumed_epoch + 1
                # ─────────────────────────────────────────────────────────────
                # FIX (A) — THE ROOT CAUSE OF THE nan% BUG.
                # The original code restored best_val_rmse_mean (a scalar) but
                # never rebuilt best_metrics_by_h, so if training crashed and
                # resumed near/at the final epoch, best_metrics_by_h stayed {}
                # for the rest of the function -> RMSE=nan -> improvement=nan%.
                # Re-run one eval pass immediately after a successful resume so
                # best_metrics_by_h always reflects the model that was actually
                # checkpointed, regardless of how many epochs remain.
                # ─────────────────────────────────────────────────────────────
                best_metrics_by_h = _eval_pass()
                logger.info(
                    f"[{fold_tag}] Resumed at epoch {resumed_epoch} — reconstructed best_metrics_by_h: "
                    + ", ".join(f"h{h} RMSE={best_metrics_by_h[h]['RMSE']:.4f}" for h in horizons)
                )
            except CheckpointShapeMismatch as e:
                logger.warning(f"[{fold_tag}] {e} Archiving stale checkpoint and starting fresh.")
                stale_path = ckpt_path + ".stale_shape_mismatch"
                if os.path.exists(stale_path):
                    os.remove(stale_path)
                os.rename(ckpt_path, stale_path)
                start_epoch, best_val_rmse_mean, epochs_no_improve = 0, float("inf"), 0
                best_metrics_by_h = {}

        n_batches = len(train_loader)
        for epoch in range(start_epoch, n_epochs):
            model.train()
            optimizer.zero_grad()
            running_loss = 0.0

            for step, (X, y_delta) in enumerate(train_loader):
                X, y_delta = X.to(DEVICE), y_delta.to(DEVICE)
                last_frame = X[:, -1, TARGET_CHANNEL_INDEX]

                pred_delta = model(X)
                loss = criterion(pred_delta, y_delta, mask, last_frame, t_std, t_mean) / grad_accum_steps
                loss.backward()
                running_loss += loss.item() * grad_accum_steps

                is_accum_boundary = ((step + 1) % grad_accum_steps == 0) or (step + 1 == n_batches)
                if is_accum_boundary:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_max_norm)
                    optimizer.step()
                    optimizer.zero_grad()
                    ema.update(model)

            scheduler.step()
            train_loss = running_loss / max(n_batches, 1)

            metrics_by_h = _eval_pass()
            val_rmse_mean = float(np.mean([metrics_by_h[h]["RMSE"] for h in horizons]))
            lr_now = optimizer.param_groups[0]["lr"]

            writer.add_scalar("Loss/train", train_loss, epoch)
            writer.add_scalar("LearningRate", lr_now, epoch)
            for h in horizons:
                for k, v in metrics_by_h[h].items():
                    if np.isfinite(v):
                        writer.add_scalar(f"Metrics_h{h}/{k}", v, epoch)

            logger.info(
                f"[{fold_tag}] epoch {epoch + 1}/{n_epochs} | train_loss={train_loss:.4f} | "
                + " | ".join(f"h{h} RMSE={metrics_by_h[h]['RMSE']:.4f}" for h in horizons)
                + f" | mean_RMSE={val_rmse_mean:.4f} | lr={lr_now:.2e}"
            )

            if val_rmse_mean < best_val_rmse_mean:
                best_val_rmse_mean = val_rmse_mean
                best_metrics_by_h = metrics_by_h
                epochs_no_improve = 0
                save_checkpoint(ckpt_path, model, optimizer, scheduler, ema, epoch, best_val_rmse_mean,
                                 extra={"hyperparams": hyperparams, "horizons": horizons, "lookback": lookback})
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    logger.info(f"[{fold_tag}] early stopping at epoch {epoch + 1} (no improvement in {patience} epochs)")
                    break

            # FIX (C): release this epoch's activation/gradient memory before the next
            # epoch's forward pass allocates fresh tensors — cheap insurance against
            # allocator fragmentation on long per-governorate runs.
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Guard: if the loop above never ran a single epoch AND resume didn't populate
        # best_metrics_by_h either (e.g. n_epochs <= start_epoch on a fresh, non-resumed
        # fold — a config error), evaluate once now rather than silently returning {}.
        if not best_metrics_by_h:
            logger.warning(f"[{fold_tag}] best_metrics_by_h was never populated — running a fallback eval pass now.")
            best_metrics_by_h = _eval_pass()

        ema_model = _build_model()
        ema_model.load_state_dict(ema.shadow)
        ema_model.eval()

        improvement_by_h = {
            h: rmse_improvement_pct_safe(
                baseline_metrics_by_h[h]["RMSE"], best_metrics_by_h.get(h, {}).get("RMSE", np.nan),
                context=f"{fold_tag} h{h}",
            )
            for h in horizons
        }
        logger.info(
            f"[{fold_tag}] FINAL - " + " | ".join(f"h{h} improvement={improvement_by_h[h]:.2f}%" for h in horizons)
        )

        result = {
            "horizons": horizons,
            "best_val_rmse_mean": best_val_rmse_mean,
            "best_metrics_by_h": best_metrics_by_h,
            "baseline_metrics_by_h": baseline_metrics_by_h,
            "rmse_improvement_by_h": improvement_by_h,
            "channel_mean": mean,
            "channel_std": std,
            "target_mask": full_ds.target_mask,
            "hyperparams": hyperparams,
            "lookback": lookback,
            "val_idx_usable": val_idx_f,
        }

        # ─────────────────────────────────────────────────────────────────────
        # FIX (C) — THE ROOT CAUSE OF THE PROGRESSIVE OOM CRASH.
        # `model` and `ema_model` are full CUDA nn.Modules. Returning them by
        # default meant every fold's models piled up, unfreed, inside
        # cv_results_direct_mh for the rest of the session. keep_models=False
        # (used by the CV loop below) drops both from the returned dict AND
        # explicitly frees their CUDA memory here, immediately, rather than
        # waiting on Python's GC to eventually notice nothing references them.
        # ─────────────────────────────────────────────────────────────────────
        if keep_models:
            result["model"] = model
            result["ema_model"] = ema_model
        else:
            _free_cuda(model, ema_model)

        return result

    finally:
        # Runs on every exit path (normal return, early return, or an exception
        # propagating out) — guarantees the writer is closed and this fold's
        # loaders/dataset copy of the governorate's region tensor are released
        # even if the fold errored out mid-training.
        writer.close()
        _free_cuda(optimizer, scheduler, ema, criterion, train_loader, val_loader, full_ds)


# =============================================================================
# Execute Moving CV across all governorates
#
# FIX (C): keep_models=False — CV folds are only ever summarized by their
# metrics afterward (see the mean_rmse_by_h aggregation below); the trained
# models themselves are never used again, so we never hold onto them.
# =============================================================================
N_EPOCHS_CV_DIRECT = 25
BATCH_SIZE_CV_DIRECT = 8
GRAD_ACCUM_STEPS_CV_DIRECT = 2
PATIENCE_CV_DIRECT = 6

DEFAULT_HYPERPARAMS_DIRECT = {
    "learning_rate": 1e-3, "weight_decay": 1e-2,
    "d_model": 64, "nhead": 4, "num_layers": 2, "dropout": 0.2,
}

cv_results_direct_mh = {}

for gov_id in range(1, N_GOVERNORATES + 1):
    gov_name = GOVERNORATE_NAMES[gov_id]
    lookback = resolve_lookback(gov_id)
    n_samples = T_TOTAL - lookback - max(HORIZONS_DIRECT) + 1
    if n_samples <= 0:
        logger.warning(f"Governorate {gov_name}: not enough history for horizons {HORIZONS_DIRECT} - skipping CV.")
        continue
    folds = generate_rolling_folds(n_samples, gap=lookback)

    fold_records = []
    for fold_i, (train_idx, val_idx) in enumerate(folds):
        fold_tag = f"gov{gov_id}_{gov_name}_direct_mh_fold{fold_i}"
        try:
            result = train_fold_direct(
                governorate_id=gov_id, train_idx=train_idx, val_idx=val_idx, lookback=lookback,
                hyperparams=resolve_hyperparams(gov_id, default=DEFAULT_HYPERPARAMS_DIRECT),
                fold_tag=fold_tag, horizons=HORIZONS_DIRECT,
                n_epochs=N_EPOCHS_CV_DIRECT, batch_size=BATCH_SIZE_CV_DIRECT,
                grad_accum_steps=GRAD_ACCUM_STEPS_CV_DIRECT, patience=PATIENCE_CV_DIRECT,
                keep_models=False,  # FIX (C): CV folds don't need the model kept around
            )
            fold_records.append(result)
        except Exception:
            logger.exception(f"[{fold_tag}] failed - skipping this fold")
        finally:
            # FIX (C): belt-and-braces cleanup between folds even on failure.
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    cv_results_direct_mh[gov_id] = fold_records
    if fold_records:
        mean_rmse_by_h = {
            h: float(np.nanmean([r["best_metrics_by_h"].get(h, {}).get("RMSE", np.nan) for r in fold_records]))
            for h in HORIZONS_DIRECT
        }
        logger.info(
            f"Governorate {gov_name}: {len(fold_records)} fold(s) completed | "
            + " | ".join(f"h{h} RMSE={mean_rmse_by_h[h]:.4f}" for h in HORIZONS_DIRECT)
        )
    _log_memory(f"after governorate {gov_name} (CV)")

# =============================================================================
# Final Production Models — train on a chronological tail split, per governorate
#
# keep_models=True here (default) is intentional and correct: unlike the CV
# loop, `ema_model` from THIS loop is what calibration/inference use
# downstream, so it must be kept. This is only 8 models total (one per
# governorate) instead of the CV loop's ~32, and it's what the rest of the
# notebook actually needs — so this is not the leak; the fix is scoped
# precisely to the CV loop above.
# =============================================================================
N_EPOCHS_FINAL_DIRECT = 60
FINAL_VAL_FRACTION_DIRECT = 0.15

final_models_direct_mh = {}

for gov_id in range(1, N_GOVERNORATES + 1):
    gov_name = GOVERNORATE_NAMES[gov_id]
    lookback = resolve_lookback(gov_id)
    n_samples = T_TOTAL - lookback - max(HORIZONS_DIRECT) + 1
    if n_samples <= 0:
        logger.warning(f"Governorate {gov_name}: not enough history for horizons {HORIZONS_DIRECT} - skipping.")
        continue

    val_size = max(1, int(round(n_samples * FINAL_VAL_FRACTION_DIRECT)))
    max_train_end = n_samples - val_size - lookback
    if max_train_end < 1:
        logger.warning(
            f"Governorate {gov_name}: not enough samples ({n_samples}) for a gap-purged final "
            f"validation split at horizons {HORIZONS_DIRECT} - skipping."
        )
        continue
    train_idx = np.arange(0, max_train_end)
    val_start = max_train_end + lookback
    val_idx = np.arange(val_start, n_samples)

    fold_tag = f"gov{gov_id}_{gov_name}_direct_mh_final"
    try:
        result = train_fold_direct(
            governorate_id=gov_id, train_idx=train_idx, val_idx=val_idx, lookback=lookback,
            hyperparams=resolve_hyperparams(gov_id, default=DEFAULT_HYPERPARAMS_DIRECT),
            fold_tag=fold_tag, horizons=HORIZONS_DIRECT,
            n_epochs=N_EPOCHS_FINAL_DIRECT, batch_size=BATCH_SIZE_CV_DIRECT,
            grad_accum_steps=GRAD_ACCUM_STEPS_CV_DIRECT, patience=10,
            keep_models=True,  # needed downstream for calibration/inference
        )
        final_models_direct_mh[gov_id] = result
        logger.info(
            f"Final direct multi-horizon model - {gov_name}: "
            + " | ".join(f"h{h} improvement={result['rmse_improvement_by_h'][h]:.2f}%" for h in HORIZONS_DIRECT)
        )
    except Exception:
        logger.exception(f"[{fold_tag}] final production training failed - skipping this governorate")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _log_memory(f"after governorate {gov_name} (final production)")

2026-08-23 19:53:05 | INFO     | NileGuard | [gov1_Aswan_direct_mh_fold0] Persistence baseline per horizon: h1: RMSE=0.2284, h3: RMSE=0.3633, h6: RMSE=0.4522, h9: RMSE=0.4669, h12: RMSE=0.4395
2026-08-23 19:53:09 | INFO     | NileGuard | [gov1_Aswan_direct_mh_fold0] epoch 1/25 | train_loss=0.5543 | h1 RMSE=0.2356 | h3 RMSE=0.3800 | h6 RMSE=0.4872 | h9 RMSE=0.5132 | h12 RMSE=0.4998 | mean_RMSE=0.4231 | lr=4.00e-04
2026-08-23 19:53:09 | INFO     | NileGuard | Checkpoint saved -> results_v2_optimized\checkpoints_v2_optimized\gov1_Aswan_direct_mh_fold0_v2_optimized.pt (epoch 0, best_val=0.42314)
2026-08-23 19:53:11 | INFO     | NileGuard | [gov1_Aswan_direct_mh_fold0] epoch 2/25 | train_loss=0.5236 | h1 RMSE=0.2943 | h3 RMSE=0.5122 | h6 RMSE=0.7220 | h9 RMSE=0.8707 | h12 RMSE=1.0033 | mean_RMSE=0.6805 | lr=6.00e-04
2026-08-23 19:53:14 | INFO     | NileGuard | [gov1_Aswan_direct_mh_fold0] epoch 3/25 | train_loss=0.4788 | h1 RMSE=0.3886 | h3 RMSE=0.7093 | h6 RMSE=1.0539 | h9 RMSE=1.3670 | h1

## 7. Evaluation, Metrics Calculation & Artifact Exporters

Fits the calibration engine on a calibration split, evaluates on the strictly disjoint report split (calibrated-with-fallback vs. uncalibrated vs. persistence baseline), ranks governorates/horizons by a composite reliability score, and exports one deployable checkpoint per governorate plus summary CSVs — all under the `_v2_optimized` suffix.

In [7]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 7: Evaluation, Metrics Calculation
# & Artifact Exporters
#
# All exported files/tables/plots/checkpoints below carry the
# `_v2_optimized` suffix so they never collide with any prior run's
# evaluation logs.
# =============================================================================

PDSI_DROUGHT_CLASSES = [
    (-float("inf"), -4.0, "Extreme Drought"), (-4.0, -3.0, "Severe Drought"),
    (-3.0, -2.0, "Moderate Drought"), (-2.0, -1.0, "Mild Drought"),
    (-1.0, 1.0, "Near Normal"), (1.0, 2.0, "Slightly Wet"),
    (2.0, 3.0, "Moderately Wet"), (3.0, 4.0, "Very Wet"), (4.0, float("inf"), "Extremely Wet"),
]


def classify_pdsi(raw_pdsi_value: float) -> str:
    """Maps a raw (denormalized) PDSI value to its Palmer-scale drought classification label."""
    for lower, upper, label in PDSI_DROUGHT_CLASSES:
        if lower <= raw_pdsi_value < upper:
            return label
    return "Unclassified"


_classify_pdsi_vec = np.vectorize(classify_pdsi, otypes=[object])


def compute_classification_accuracy(y_true_raw, y_pred_raw) -> float:
    """Percentage of pixels whose Palmer drought-severity class matches between true and predicted PDSI."""
    y_true_raw = np.asarray(y_true_raw, dtype=np.float64).ravel()
    y_pred_raw = np.asarray(y_pred_raw, dtype=np.float64).ravel()
    if y_true_raw.size == 0:
        return float("nan")
    true_classes = _classify_pdsi_vec(y_true_raw)
    pred_classes = _classify_pdsi_vec(y_pred_raw)
    return float(np.mean(true_classes == pred_classes) * 100.0)


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot: int = 1000, ci: float = 0.95, seed: int = SEED):
    """Percentile-bootstrap confidence interval for an arbitrary pixel-level metric function.

    Args:
        y_true: Raw true values.
        y_pred: Raw predicted values.
        metric_fn: Callable ``metric_fn(y_true, y_pred) -> float``.
        n_boot: Number of bootstrap resamples.
        ci: Confidence level (e.g. 0.95).
        seed: RNG seed for reproducibility.

    Returns:
        A ``(lo, hi)`` tuple bounding the ``ci``-level confidence interval.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    n = y_true.size
    if n == 0:
        return float("nan"), float("nan")
    stats = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[b] = metric_fn(y_true[idx], y_pred[idx])
    alpha = (1.0 - ci) / 2.0
    lo = float(np.nanpercentile(stats, alpha * 100.0))
    hi = float(np.nanpercentile(stats, (1.0 - alpha) * 100.0))
    return lo, hi


def _direct_model_predict(model: nn.Module, full_ds: "NileGuardDirectMultiHorizonDataset",
                           sample_indices: list, mask_np: np.ndarray, t_mean: float, t_std: float,
                           device, batch_size: int = 16):
    """Runs one forward pass per batch (no autoregressive feedback) and returns raw true/pred arrays per horizon.

    Args:
        model: A trained ``NileGuardDirectMultiHorizonModel``.
        full_ds: The governorate's ``NileGuardDirectMultiHorizonDataset``.
        sample_indices: Sample-start indices to predict on.
        mask_np: (H, W) boolean array of real governorate pixels.
        t_mean: PDSI channel mean (for de-normalization).
        t_std: PDSI channel std (for de-normalization).
        device: Torch device.
        batch_size: Batch size for inference.

    Returns:
        Two dicts, ``(y_true_by_h, y_pred_by_h)``, each keyed by horizon
        and holding concatenated flat raw (denormalized) arrays.
    """
    horizons = full_ds.horizons
    y_true_by_h = {h: [] for h in horizons}
    y_pred_by_h = {h: [] for h in horizons}

    model.eval()
    with torch.no_grad():
        for start in range(0, len(sample_indices), batch_size):
            batch_ids = sample_indices[start:start + batch_size]
            X = torch.stack([full_ds.region[i: i + full_ds.lookback] for i in batch_ids]).to(device)
            last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
            pred_delta = model(X)

            for i, h in enumerate(horizons):
                future_idx_list = [j + full_ds.lookback - 1 + h for j in batch_ids]
                y_true_abs = torch.stack(
                    [full_ds.region[fi, TARGET_CHANNEL_INDEX] for fi in future_idx_list]
                ).to(device)
                y_pred_abs = last_frame + pred_delta[:, i]

                y_true_by_h[h].append((y_true_abs[:, mask_np] * t_std + t_mean).cpu().numpy())
                y_pred_by_h[h].append((y_pred_abs[:, mask_np] * t_std + t_mean).cpu().numpy())

    y_true_by_h = {h: np.concatenate(v) if v else np.array([]) for h, v in y_true_by_h.items()}
    y_pred_by_h = {h: np.concatenate(v) if v else np.array([]) for h, v in y_pred_by_h.items()}
    return y_true_by_h, y_pred_by_h


def fit_governorate_horizon_calibration(final_models: dict, calib_fraction: float = 0.5) -> "HorizonSpecificCalibrator":
    """Fits the per-(governorate, horizon) Isotonic Regression calibrator with strict fallback.

    Splits each governorate's held-out ``val_idx_usable`` chronologically
    into a CALIBRATION half (earlier) and a REPORT half (later, strictly
    after — no leakage), runs one direct forward pass over the
    calibration half, and fits `HorizonSpecificCalibrator` from it. The
    fallback decision (accept/reject Isotonic Regression) is made entirely
    on the calibration half; the report half is reserved for Section 7's
    final evaluation numbers.

    Args:
        final_models: ``{governorate_id: train_fold_direct(...) result}``.
        calib_fraction: Fraction of each governorate's held-out
            validation samples used for calibration (the remainder is
            the disjoint report split).

    Returns:
        A fitted `HorizonSpecificCalibrator`.
    """
    calibrator = HorizonSpecificCalibrator()

    for gov_id, info in final_models.items():
        gov_name = GOVERNORATE_NAMES[gov_id]
        lookback = info["lookback"]
        mean, std = info["channel_mean"], info["channel_std"]
        t_mean, t_std = float(mean[TARGET_CHANNEL_INDEX]), float(std[TARGET_CHANNEL_INDEX])
        full_ds = NileGuardDirectMultiHorizonDataset(master_tensor, gov_id, lookback, mean, std, horizons=info["horizons"])
        mask_np = info["target_mask"].numpy() if torch.is_tensor(info["target_mask"]) else info["target_mask"]
        model = info["ema_model"]

        val_idx = sorted(info["val_idx_usable"].tolist())
        n_calib = max(1, int(round(len(val_idx) * calib_fraction)))
        calib_ids, report_ids = val_idx[:n_calib], val_idx[n_calib:]
        if not calib_ids or not report_ids:
            logger.warning(f"[Calibration] {gov_name}: not enough held-out samples to split calib/report - skipping.")
            continue

        y_true_by_h, y_pred_by_h = _direct_model_predict(model, full_ds, calib_ids, mask_np, t_mean, t_std, DEVICE)
        for h in info["horizons"]:
            accepted = calibrator.fit(gov_id, h, y_true_by_h[h], y_pred_by_h[h])
            status = "ACCEPTED" if accepted else "REJECTED (fallback to uncalibrated)"
            logger.info(
                f"[Calibration] {gov_name} h={h}mo: Isotonic Regression {status} "
                f"(n={len(y_true_by_h[h])}, RMSE_uncal={calibrator.rmse_before[(gov_id, h)]:.4f}, "
                f"RMSE_after_rule={calibrator.rmse_after[(gov_id, h)]:.4f})"
            )

    return calibrator


governorate_horizon_calibrator = fit_governorate_horizon_calibration(final_models_direct_mh)
calibration_table_df = governorate_horizon_calibrator.to_dataframe()
calibration_table_df


def evaluate_direct_multi_horizon(final_models: dict, calibrator: "HorizonSpecificCalibrator",
                                   calib_fraction: float = 0.5) -> pd.DataFrame:
    """Full evaluation on the REPORT split (disjoint from the calibration split).

    Reports calibrated (with strict fallback applied), uncalibrated, and
    persistence-baseline metrics side by side for every
    (governorate, horizon) pair, plus which calibration decision was
    actually used.

    Args:
        final_models: ``{governorate_id: train_fold_direct(...) result}``.
        calibrator: A fitted `HorizonSpecificCalibrator`.
        calib_fraction: Must match the fraction used in
            `fit_governorate_horizon_calibration` so the report split
            here is exactly the complementary, disjoint half.

    Returns:
        A tidy ``pandas.DataFrame``, one row per (governorate, horizon).
    """
    rows = []
    for gov_id, info in final_models.items():
        gov_name = GOVERNORATE_NAMES[gov_id]
        lookback = info["lookback"]
        mean, std = info["channel_mean"], info["channel_std"]
        t_mean, t_std = float(mean[TARGET_CHANNEL_INDEX]), float(std[TARGET_CHANNEL_INDEX])
        full_ds = NileGuardDirectMultiHorizonDataset(master_tensor, gov_id, lookback, mean, std, horizons=info["horizons"])
        mask_np = info["target_mask"].numpy() if torch.is_tensor(info["target_mask"]) else info["target_mask"]
        model = info["ema_model"]

        val_idx = sorted(info["val_idx_usable"].tolist())
        n_calib = max(1, int(round(len(val_idx) * calib_fraction)))
        report_ids = val_idx[n_calib:]
        if not report_ids:
            logger.warning(f"[MultiHorizonEval] {gov_name}: no report-split samples - skipping.")
            continue

        y_true_by_h, y_pred_uncal_by_h = _direct_model_predict(model, full_ds, report_ids, mask_np, t_mean, t_std, DEVICE)

        for h in info["horizons"]:
            y_true = y_true_by_h[h]
            y_pred_uncal = y_pred_uncal_by_h[h]
            y_pred_reported = calibrator.apply(gov_id, h, y_pred_uncal)  # calibrated, or uncalibrated fallback
            calibration_was_accepted = calibrator.accepted.get((gov_id, h), False)

            last_frame_report = [
                (full_ds.region[i + lookback - 1, TARGET_CHANNEL_INDEX] * t_std + t_mean).numpy()[mask_np]
                for i in report_ids
            ]
            y_base = np.concatenate(last_frame_report) if last_frame_report else np.array([])

            model_metrics = compute_metrics(y_true, y_pred_reported)
            uncal_metrics = compute_metrics(y_true, y_pred_uncal)
            baseline_metrics = compute_metrics(y_true, y_base)
            improvement = rmse_improvement_pct_safe(baseline_metrics["RMSE"], model_metrics["RMSE"])
            uncal_improvement = rmse_improvement_pct_safe(baseline_metrics["RMSE"], uncal_metrics["RMSE"])

            rows.append({
                "Governorate": gov_name, "Horizon_Months": h, "N_Samples": int(y_true.size),
                "Calibration_Applied": calibration_was_accepted,
                "Reported_RMSE": model_metrics["RMSE"], "Uncalibrated_RMSE": uncal_metrics["RMSE"],
                "Baseline_RMSE": baseline_metrics["RMSE"],
                "RMSE_Improvement_%_vs_Baseline": improvement,
                "RMSE_Improvement_%_vs_Baseline_Uncalibrated": uncal_improvement,
                "Reported_MAE": model_metrics["MAE"], "Baseline_MAE": baseline_metrics["MAE"],
                "Reported_R2": model_metrics["R2"], "Baseline_R2": baseline_metrics["R2"],
                "Reported_NSE": model_metrics["NSE"], "Baseline_NSE": baseline_metrics["NSE"],
                "Reported_Bias": model_metrics["Bias"], "Baseline_Bias": baseline_metrics["Bias"],
                "Reported_ClassAcc_%": compute_classification_accuracy(y_true, y_pred_reported),
                "Baseline_ClassAcc_%": compute_classification_accuracy(y_true, y_base),
            })

        logger.info(
            f"[MultiHorizonEval] {gov_name}: "
            + " | ".join(
                f"h{h} RMSE={next(r['Reported_RMSE'] for r in rows if r['Governorate'] == gov_name and r['Horizon_Months'] == h):.4f}"
                for h in info["horizons"]
            )
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["Governorate", "Horizon_Months"]).reset_index(drop=True)
    return df


direct_multi_horizon_df = evaluate_direct_multi_horizon(final_models_direct_mh, governorate_horizon_calibrator)

logger.info("=" * 90)
logger.info("DIRECT MULTI-HORIZON EVALUATION (h=1,3,6,9,12) — Calibrated-with-Fallback vs. Persistence Baseline")
logger.info("=" * 90)
direct_multi_horizon_df


# =============================================================================
# Best Governorate & Best Horizon Analytical Summary
# =============================================================================
def _minmax_normalize(series: pd.Series, higher_is_better: bool) -> pd.Series:
    """Maps a metric column to [0, 1] where 1 is always 'better'."""
    s = series.astype(float)
    lo, hi = np.nanmin(s), np.nanmax(s)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return pd.Series(0.5, index=s.index)
    norm = (s - lo) / (hi - lo)
    return norm if higher_is_better else (1.0 - norm)


def summarize_best_governorate(df: pd.DataFrame) -> pd.DataFrame:
    """Ranks governorates by a composite score (RMSE + R2 + ClassAcc) averaged across all horizons."""
    agg = df.groupby("Governorate").agg(
        Mean_RMSE=("Reported_RMSE", "mean"), Mean_MAE=("Reported_MAE", "mean"),
        Mean_R2=("Reported_R2", "mean"), Mean_NSE=("Reported_NSE", "mean"),
        Mean_ClassAcc_pct=("Reported_ClassAcc_%", "mean"),
        Mean_RMSE_Improvement_pct=("RMSE_Improvement_%_vs_Baseline", "mean"),
    ).reset_index()

    agg["_score_rmse"] = _minmax_normalize(agg["Mean_RMSE"], higher_is_better=False)
    agg["_score_r2"] = _minmax_normalize(agg["Mean_R2"], higher_is_better=True)
    agg["_score_classacc"] = _minmax_normalize(agg["Mean_ClassAcc_pct"], higher_is_better=True)
    agg["Composite_Score"] = (agg["_score_rmse"] + agg["_score_r2"] + agg["_score_classacc"]) / 3.0
    agg = agg.drop(columns=["_score_rmse", "_score_r2", "_score_classacc"])
    return agg.sort_values("Composite_Score", ascending=False).reset_index(drop=True)


def summarize_best_horizon_per_governorate(df: pd.DataFrame) -> pd.DataFrame:
    """For each governorate, returns its single most reliable forecast horizon by composite score."""
    rows = []
    for gov_name, gdf in df.groupby("Governorate"):
        gdf = gdf.copy()
        gdf["_score_rmse"] = _minmax_normalize(gdf["Reported_RMSE"], higher_is_better=False)
        gdf["_score_r2"] = _minmax_normalize(gdf["Reported_R2"], higher_is_better=True)
        gdf["_score_classacc"] = _minmax_normalize(gdf["Reported_ClassAcc_%"], higher_is_better=True)
        gdf["Composite_Score"] = (gdf["_score_rmse"] + gdf["_score_r2"] + gdf["_score_classacc"]) / 3.0
        best = gdf.sort_values("Composite_Score", ascending=False).iloc[0]
        rows.append({
            "Governorate": gov_name, "Best_Horizon_Months": int(best["Horizon_Months"]),
            "RMSE": best["Reported_RMSE"], "R2": best["Reported_R2"], "ClassAcc_%": best["Reported_ClassAcc_%"],
            "RMSE_Improvement_%_vs_Baseline": best["RMSE_Improvement_%_vs_Baseline"],
            "Composite_Score": best["Composite_Score"],
        })
    return pd.DataFrame(rows).sort_values("Composite_Score", ascending=False).reset_index(drop=True)


best_governorate_summary_df = summarize_best_governorate(direct_multi_horizon_df)
best_horizon_per_governorate_df = summarize_best_horizon_per_governorate(direct_multi_horizon_df)

logger.info("=" * 90)
logger.info("BEST GOVERNORATE — overall forecast reliability across h=1,3,6,9,12")
logger.info("=" * 90)
if not best_governorate_summary_df.empty:
    top = best_governorate_summary_df.iloc[0]
    logger.info(
        f"Most reliable governorate overall : {top['Governorate']} "
        f"(mean RMSE={top['Mean_RMSE']:.4f}, mean R2={top['Mean_R2']:.4f}, "
        f"mean ClassAcc={top['Mean_ClassAcc_pct']:.2f}%, composite={top['Composite_Score']:.4f})"
    )

print("\n=== Best Governorate (overall, across all horizons) ===")
display(best_governorate_summary_df)
print("\n=== Best Forecast Horizon per Governorate ===")
display(best_horizon_per_governorate_df)


# =============================================================================
# Artifact Exporters — everything written here carries the _v2_optimized suffix
# =============================================================================
def export_production_artifacts(final_models: dict, calibrator: "HorizonSpecificCalibrator",
                                 eval_df: pd.DataFrame, artifacts_dir: str = None) -> None:
    """Exports one deployable checkpoint per governorate plus summary CSVs.

    Args:
        final_models: ``{governorate_id: train_fold_direct(...) result}``.
        calibrator: The fitted `HorizonSpecificCalibrator` used at report time.
        eval_df: The output of `evaluate_direct_multi_horizon`.
        artifacts_dir: Output directory (defaults to the global ``ARTIFACTS_DIR``).
    """
    artifacts_dir = artifacts_dir or str(ARTIFACTS_DIR)
    os.makedirs(artifacts_dir, exist_ok=True)

    for gov_id, info in final_models.items():
        gov_name = GOVERNORATE_NAMES[gov_id]
        artifact_path = os.path.join(artifacts_dir, f"model_final_{gov_name}_v2_optimized.pth")

        gov_eval = eval_df[eval_df["Governorate"] == gov_name].to_dict("records") if not eval_df.empty else []

        torch.save({
            "model_state": info["ema_model"].state_dict(),
            "lookback": info["lookback"],  # == LOCKED LOOKBACK for every governorate
            "horizons": info["horizons"],
            "hyperparams": info["hyperparams"],
            "channel_mean": info["channel_mean"],
            "channel_std": info["channel_std"],
            "target_mask": info["target_mask"],
            "bbox": GOV_BBOXES[gov_id],
            "in_channels": int(master_tensor.shape[-1]),
            "feature_names": FEATURE_NAMES,
            "target_channel_index": TARGET_CHANNEL_INDEX,
            "gov_mask_channel_index": GOV_MASK_CHANNEL_INDEX,
            "architecture": "NileGuardDirectMultiHorizonModel_v2_optimized",
            "validation_metrics_by_horizon": info["best_metrics_by_h"],
            "baseline_metrics_by_horizon": info["baseline_metrics_by_h"],
            "rmse_improvement_by_horizon": info["rmse_improvement_by_h"],
            "calibration_table": calibrator.to_dataframe().to_dict("records"),
            "multi_horizon_report_metrics": gov_eval,
        }, artifact_path)

        logger.info(f"Exported deployment artifact -> {artifact_path}")

    calibration_table_df.to_csv(os.path.join(artifacts_dir, "calibration_table_v2_optimized.csv"), index=False)
    eval_df.to_csv(os.path.join(artifacts_dir, "direct_multi_horizon_eval_v2_optimized.csv"), index=False)
    best_governorate_summary_df.to_csv(os.path.join(artifacts_dir, "best_governorate_summary_v2_optimized.csv"), index=False)
    best_horizon_per_governorate_df.to_csv(os.path.join(artifacts_dir, "best_horizon_per_governorate_v2_optimized.csv"), index=False)

    logger.info(f"All production artifacts saved under ./{artifacts_dir}/")


export_production_artifacts(final_models_direct_mh, governorate_horizon_calibrator, direct_multi_horizon_df)


2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Aswan h=1mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.2967, RMSE_after_rule=0.2759)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Aswan h=3mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.5069, RMSE_after_rule=0.4365)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Aswan h=6mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.6560, RMSE_after_rule=0.5161)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Aswan h=9mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.7542, RMSE_after_rule=0.5382)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Aswan h=12mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.8100, RMSE_after_rule=0.5470)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Asyut h=1mo: Isotonic Regression ACCEPTED (n=55, RMSE_uncal=0.2574, RMSE_after_rule=0.2344)
2026-08-23 20:52:24 | INFO     | NileGuard | [Calibration] Asyut h=3mo: Isotonic Regression A


=== Best Governorate (overall, across all horizons) ===


,Governorate,Mean_RMSE,Mean_MAE,Mean_R2,Mean_NSE,Mean_ClassAcc_pct,Mean_RMSE_Improvement_pct,Composite_Score
0,Fayoum,0.585538,0.421367,0.222894,0.222894,88.405889,11.561994,0.947235
1,Qena,0.867346,0.677176,0.263812,0.263812,69.690191,7.687333,0.619658
2,Aswan,0.824368,0.585404,0.111812,0.111812,75.077922,10.721549,0.509835
3,Asyut,0.871985,0.585032,0.052386,0.052386,79.696970,9.369266,0.455491
4,Minya,0.895650,0.637920,0.005320,0.005320,75.929334,11.936751,0.338432
5,BeniSuef,0.757639,0.622354,0.049742,0.049742,62.077922,7.131665,0.325957
6,Luxor,1.123541,0.827965,0.169466,0.169466,67.654356,6.769031,0.318835
7,Sohag,1.131161,0.841076,0.213107,0.213107,58.438596,3.353173,0.267948



=== Best Forecast Horizon per Governorate ===


,Governorate,Best_Horizon_Months,RMSE,R2,ClassAcc_%,RMSE_Improvement_%_vs_Baseline,Composite_Score
0,Aswan,1,0.452010,0.730024,89.272727,-0.423759,1.0
1,Asyut,1,0.447425,0.738753,91.363636,0.772370,1.0
2,BeniSuef,1,0.451291,0.635680,86.753247,-4.939555,1.0
3,Fayoum,1,0.310406,0.793497,94.582426,0.919088,1.0
4,Luxor,1,0.527158,0.821483,86.197917,-1.596006,1.0
5,Minya,1,0.495564,0.668506,89.348546,-0.192312,1.0
6,Qena,1,0.478344,0.788941,83.032297,-2.827565,1.0
7,Sohag,1,0.554451,0.839711,83.851675,-1.222089,1.0


2026-08-23 20:52:29 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_Aswan_v2_optimized.pth
2026-08-23 20:52:29 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_Asyut_v2_optimized.pth
2026-08-23 20:52:29 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_BeniSuef_v2_optimized.pth
2026-08-23 20:52:29 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_Fayoum_v2_optimized.pth
2026-08-23 20:52:30 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_Luxor_v2_optimized.pth
2026-08-23 20:52:30 | INFO     | NileGuard | Exported deployment artifact -> results_v2_optimized\artifacts_v2_optimized\model_final_Minya_v2_optimized.pth
2026-08-23 20:52:30 | INFO     | NileGuard | Exported deploy

## 8. Transfer Learning / Fine-Tuning Pipeline, Deployment Weight Export & Power BI CSV Export

In [8]:
# =============================================================================
# NileGuard v17 (v2_optimized) — Section 8: Transfer Learning / Fine-Tuning
# Pipeline, Deployment Weight Export & Power BI CSV Export
#
# Loads each governorate's already-trained `_v2_optimized` production
# checkpoint (Section 7's `export_production_artifacts` output), FREEZES the
# CNN spatial encoder and the Divided Space-Time Transformer encoder blocks
# (`param.requires_grad = False`), keeps ONLY the multi-horizon prediction
# heads trainable, fine-tunes at a low learning rate with early stopping,
# exports one consolidated deployment weights file, and writes tidy
# per-horizon prediction + metrics CSVs formatted for Power BI.
# =============================================================================

# -----------------------------------------------------------------------
# 8.0 Directories & Fine-Tuning Config
# -----------------------------------------------------------------------
TRANSFER_DIR = PROJECT_DIR / "results_v2_transfer"
TRANSFER_CHECKPOINT_DIR = TRANSFER_DIR / "checkpoints_transfer"
DEPLOYMENT_WEIGHTS_DIR = TRANSFER_DIR / "deployment_weights"
DASHBOARD_EXPORTS_DIR = TRANSFER_DIR / "dashboard_exports"

for directory in [TRANSFER_DIR, TRANSFER_CHECKPOINT_DIR, DEPLOYMENT_WEIGHTS_DIR, DASHBOARD_EXPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PRETRAINED_ARTIFACTS_DIR = ARTIFACTS_DIR  # Section 7 output: model_final_<Gov>_v2_optimized.pth
FINE_TUNE_LR = 1e-4
FINE_TUNE_WEIGHT_DECAY = 1e-2
FINE_TUNE_MAX_EPOCHS = 30
FINE_TUNE_PATIENCE = 6
FINE_TUNE_BATCH_SIZE = 8
FINE_TUNE_VAL_FRACTION = 0.15

logger.info("=" * 78)
logger.info("NileGuard v2_optimized -> v2_transfer — Section 8: Transfer Learning")
logger.info("=" * 78)
logger.info(f"Pretrained artifacts source : {PRETRAINED_ARTIFACTS_DIR}")
logger.info(f"Fine-tuning learning rate   : {FINE_TUNE_LR}")
logger.info(f"Deployment weights output   : {DEPLOYMENT_WEIGHTS_DIR}")
logger.info(f"Dashboard CSV output        : {DASHBOARD_EXPORTS_DIR}")


# -----------------------------------------------------------------------
# 8.1 Load a pretrained governorate checkpoint & rebuild its model
# -----------------------------------------------------------------------
def load_pretrained_governorate_model(governorate_id: int, artifacts_dir=None):
    """Loads one governorate's `_v2_optimized` production checkpoint and rebuilds the model.

    Args:
        governorate_id: Governorate ID (1-8).
        artifacts_dir: Directory containing `model_final_<Gov>_v2_optimized.pth`
            files, as written by Section 7's `export_production_artifacts`.
            Defaults to the global `ARTIFACTS_DIR`.

    Returns:
        A ``(model, checkpoint_dict)`` tuple. ``model`` is on ``DEVICE``,
        in eval mode, with the pretrained weights already loaded.
        ``checkpoint_dict`` is the full raw checkpoint (hyperparams,
        normalization stats, bbox, prior metrics, ...).

    Raises:
        FileNotFoundError: If no matching checkpoint file exists at the
            expected path.
    """
    artifacts_dir = Path(artifacts_dir or ARTIFACTS_DIR)
    gov_name = GOVERNORATE_NAMES[governorate_id]
    ckpt_path = artifacts_dir / f"model_final_{gov_name}_v2_optimized.pth"
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"No pretrained `_v2_optimized` checkpoint found for {gov_name} at {ckpt_path}. "
            "Run Section 7's `export_production_artifacts(...)` first, or pass a different `artifacts_dir`."
        )

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    r0, r1, c0, c1 = ckpt["bbox"]

    model = NileGuardDirectMultiHorizonModel(
        lookback=ckpt["lookback"], in_channels=ckpt["in_channels"],
        height=r1 - r0, width=c1 - c0, horizons=ckpt["horizons"],
        d_model=ckpt["hyperparams"].get("d_model", 64), nhead=ckpt["hyperparams"].get("nhead", 4),
        num_spatiotemporal_layers=ckpt["hyperparams"].get("num_layers", 2),
        dropout=ckpt["hyperparams"].get("dropout", 0.2), verbose=False,
    ).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    logger.info(f"[TransferLearning] Loaded pretrained weights for {gov_name} <- {ckpt_path}")
    return model, ckpt


# -----------------------------------------------------------------------
# 8.2 Freeze backbone, keep multi-horizon prediction heads trainable
# -----------------------------------------------------------------------
def freeze_backbone_keep_heads_trainable(model: "NileGuardDirectMultiHorizonModel") -> dict:
    """Freezes the CNN spatial encoder and Transformer encoder blocks; keeps the multi-horizon heads trainable.

    Frozen (``param.requires_grad = False``):
        - ``model.spatial_encoder`` — early CNN spatial layers (multi-scale
          conv blocks, SE block, feature fusion).
        - ``model.st_attention`` — spatio-temporal attention.
        - ``model.spatial_pos_embed``, ``model.temporal_pos_encoding`` —
          positional embeddings.
        - ``model.st_blocks`` — Divided Space-Time Transformer encoder layers.
        - ``model.final_norm``.

    Trainable (``param.requires_grad = True``):
        - ``model.temporal_attn_scorer`` — temporal-attention pooling.
        - ``model.decoder_proj``.
        - ``model.multi_horizon_decoder`` — the multi-horizon prediction heads.

    Args:
        model: A `NileGuardDirectMultiHorizonModel` instance, modified in place.

    Returns:
        A dict with frozen/trainable parameter counts and module-name
        lists, for logging and for the deployment-weights export.
    """
    frozen_modules = [
        model.spatial_encoder, model.st_attention, model.temporal_pos_encoding,
        model.st_blocks, model.final_norm,
    ]
    frozen_module_names = ["spatial_encoder", "st_attention", "temporal_pos_encoding", "st_blocks", "final_norm"]
    for module in frozen_modules:
        for param in module.parameters():
            param.requires_grad = False
    model.spatial_pos_embed.requires_grad = False  # bare nn.Parameter, not a sub-module

    trainable_modules = [model.temporal_attn_scorer, model.decoder_proj, model.multi_horizon_decoder]
    trainable_module_names = ["temporal_attn_scorer", "decoder_proj", "multi_horizon_decoder"]
    for module in trainable_modules:
        for param in module.parameters():
            param.requires_grad = True

    n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(
        f"[TransferLearning] Frozen: {frozen_module_names} ({n_frozen:,} params) | "
        f"Trainable heads: {trainable_module_names} ({n_trainable:,} params)"
    )
    return {
        "frozen_module_names": frozen_module_names,
        "trainable_module_names": trainable_module_names,
        "n_frozen_params": n_frozen,
        "n_trainable_params": n_trainable,
    }


# -----------------------------------------------------------------------
# 8.3 Shared eval helpers (generalized versions of Section 6's inline closures)
# -----------------------------------------------------------------------
def evaluate_direct_model_on_loader(model: nn.Module, loader: DataLoader, mask: torch.Tensor,
                                     horizons: list, t_mean: float, t_std: float,
                                     device=DEVICE) -> dict:
    """Runs one full evaluation pass over `loader` and returns per-horizon metrics.

    Args:
        model: A `NileGuardDirectMultiHorizonModel` in eval mode.
        loader: Yields ``(X, y_delta)`` batches from a
            `NileGuardDirectMultiHorizonDataset`.
        mask: (H, W) bool tensor of real governorate pixels, on `device`.
        horizons: Forecast horizons in months.
        t_mean: PDSI channel mean (for de-normalization).
        t_std: PDSI channel std (for de-normalization).
        device: Torch device.

    Returns:
        ``{horizon: compute_metrics(...) dict}``.
    """
    model.eval()
    y_true_by_h = {h: [] for h in horizons}
    y_pred_by_h = {h: [] for h in horizons}
    with torch.no_grad():
        for X, y_delta in loader:
            X, y_delta = X.to(device), y_delta.to(device)
            last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
            pred_delta = model(X)
            for i, h in enumerate(horizons):
                pred_abs = last_frame + pred_delta[:, i]
                y_abs = last_frame + y_delta[:, i]
                y_true_by_h[h].append((y_abs[:, mask] * t_std + t_mean).cpu().numpy())
                y_pred_by_h[h].append((pred_abs[:, mask] * t_std + t_mean).cpu().numpy())
    out = {}
    for h in horizons:
        yt = np.concatenate(y_true_by_h[h]) if y_true_by_h[h] else np.array([])
        yp = np.concatenate(y_pred_by_h[h]) if y_pred_by_h[h] else np.array([])
        out[h] = compute_metrics(yt, yp)
    return out


def compute_persistence_baseline_metrics(loader: DataLoader, mask: torch.Tensor, horizons: list,
                                          t_mean: float, t_std: float, device=DEVICE) -> dict:
    """Computes persistence-baseline (predicted delta = 0) metrics per horizon over `loader`."""
    baseline_true = {h: [] for h in horizons}
    baseline_pred = {h: [] for h in horizons}
    with torch.no_grad():
        for X, y_delta in loader:
            X, y_delta = X.to(device), y_delta.to(device)
            last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
            for i, h in enumerate(horizons):
                y_abs = last_frame + y_delta[:, i]
                baseline_true[h].append((y_abs[:, mask] * t_std + t_mean).cpu().numpy())
                baseline_pred[h].append((last_frame[:, mask] * t_std + t_mean).cpu().numpy())
    out = {}
    for h in horizons:
        yt = np.concatenate(baseline_true[h]) if baseline_true[h] else np.array([])
        yp = np.concatenate(baseline_pred[h]) if baseline_pred[h] else np.array([])
        out[h] = compute_metrics(yt, yp)
    return out


# -----------------------------------------------------------------------
# 8.4 Fine-tuning loop for one governorate
# -----------------------------------------------------------------------
def fine_tune_fold_direct(
    governorate_id: int, hyperparams: dict = None, artifacts_dir=None,
    val_fraction: float = FINE_TUNE_VAL_FRACTION, horizons: list = None,
    n_epochs: int = FINE_TUNE_MAX_EPOCHS, batch_size: int = FINE_TUNE_BATCH_SIZE,
    patience: int = FINE_TUNE_PATIENCE, learning_rate: float = FINE_TUNE_LR,
    weight_decay: float = FINE_TUNE_WEIGHT_DECAY, grad_clip_max_norm: float = 1.0,
    checkpoint_dir=None, resume: bool = True,
) -> dict:
    """Fine-tunes one governorate's pretrained `NileGuardDirectMultiHorizonModel` with a frozen backbone.

    Loads the `_v2_optimized` production checkpoint for `governorate_id`
    (Section 7), freezes the CNN spatial encoder and Transformer encoder
    blocks, keeps the multi-horizon prediction heads trainable, and
    fine-tunes on a fresh chronological tail split (`val_fraction`) with
    an AdamW optimizer restricted to the trainable parameters and
    early stopping on mean validation RMSE across horizons.

    Args:
        governorate_id: Governorate ID (1-8).
        hyperparams: Architecture hyperparameters (`d_model`, `nhead`,
            `num_layers`, `dropout`). Defaults to the checkpoint's own
            saved hyperparameters, so the rebuilt model matches exactly.
        artifacts_dir: Directory holding the pretrained `_v2_optimized`
            checkpoints (defaults to `ARTIFACTS_DIR`).
        val_fraction: Fraction of usable samples held out chronologically
            (tail) for fine-tuning validation / early stopping.
        horizons: Forecast horizons in months (defaults to the checkpoint's
            own horizons).
        n_epochs: Maximum fine-tuning epochs.
        batch_size: Mini-batch size.
        patience: Early-stopping patience, in epochs without improvement.
        learning_rate: Fine-tuning optimizer learning rate (LOCKED to
            ``1e-4`` by default, per the transfer-learning spec).
        weight_decay: AdamW weight decay for the trainable head parameters.
        grad_clip_max_norm: Gradient-norm clipping threshold.
        checkpoint_dir: Directory for fine-tuning checkpoints (defaults to
            `TRANSFER_CHECKPOINT_DIR`).
        resume: If True, resumes from an existing fine-tuning checkpoint
            when shape-compatible.

    Returns:
        A dict shaped like Section 6's `train_fold_direct` result
        (`model`, `ema_model`, `best_metrics_by_h`, `baseline_metrics_by_h`,
        `rmse_improvement_by_h`, `channel_mean`, `channel_std`,
        `target_mask`, `hyperparams`, `lookback`, `horizons`, `bbox`,
        `in_channels`, `val_idx_usable`) plus `freeze_info`, so it plugs
        directly into the same evaluation/export helpers used for the
        original production models.

    Raises:
        FileNotFoundError: If no pretrained checkpoint exists for this governorate.
        ValueError: If there are not enough samples for the requested `val_fraction`.
    """
    gov_name = GOVERNORATE_NAMES[governorate_id]
    model, ckpt = load_pretrained_governorate_model(governorate_id, artifacts_dir)
    freeze_info = freeze_backbone_keep_heads_trainable(model)

    lookback = ckpt["lookback"]
    horizons = sorted(horizons if horizons is not None else ckpt["horizons"])
    mean, std = ckpt["channel_mean"], ckpt["channel_std"]
    t_mean, t_std = float(mean[TARGET_CHANNEL_INDEX]), float(std[TARGET_CHANNEL_INDEX])
    hyperparams = hyperparams or ckpt["hyperparams"]

    full_ds = NileGuardDirectMultiHorizonDataset(master_tensor, governorate_id, lookback, mean, std, horizons=horizons)
    mask = full_ds.target_mask.to(DEVICE)

    n_samples = len(full_ds)
    val_size = max(1, int(round(n_samples * val_fraction)))
    train_end = n_samples - val_size
    if train_end < 1:
        raise ValueError(
            f"[{gov_name}] not enough samples ({n_samples}) for a {val_fraction:.0%} fine-tuning validation split."
        )
    train_idx = np.arange(0, train_end)
    val_idx = np.arange(train_end, n_samples)

    train_loader = DataLoader(Subset(full_ds, train_idx.tolist()), batch_size=batch_size,
                               shuffle=True, drop_last=True)
    val_loader = DataLoader(Subset(full_ds, val_idx.tolist()), batch_size=batch_size, shuffle=False)

    baseline_metrics_by_h = compute_persistence_baseline_metrics(val_loader, mask, horizons, t_mean, t_std)
    logger.info(
        f"[TransferLearning:{gov_name}] Persistence baseline: "
        + ", ".join(f"h{h}: RMSE={baseline_metrics_by_h[h]['RMSE']:.4f}" for h in horizons)
    )

    criterion = HorizonWeightedHuberLoss(horizons).to(DEVICE)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
    ema = EMA(model, decay=0.999)

    checkpoint_dir = checkpoint_dir or str(TRANSFER_CHECKPOINT_DIR)
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f"gov{governorate_id}_{gov_name}_transfer.pt")

    start_epoch, best_val_rmse_mean, epochs_no_improve = 0, float("inf"), 0
    best_metrics_by_h: dict = {}

    if resume and os.path.exists(ckpt_path):
        try:
            resumed_epoch, best_val_rmse_mean, _ = load_checkpoint(ckpt_path, model, optimizer, None, ema)
            start_epoch = resumed_epoch + 1
            best_metrics_by_h = evaluate_direct_model_on_loader(model, val_loader, mask, horizons, t_mean, t_std)
            logger.info(f"[TransferLearning:{gov_name}] Resumed fine-tuning from epoch {resumed_epoch}.")
        except CheckpointShapeMismatch as e:
            logger.warning(f"[TransferLearning:{gov_name}] {e} Starting fine-tuning fresh.")

    for epoch in range(start_epoch, n_epochs):
        model.train()
        running_loss = 0.0
        for X, y_delta in train_loader:
            X, y_delta = X.to(DEVICE), y_delta.to(DEVICE)
            last_frame = X[:, -1, TARGET_CHANNEL_INDEX]
            optimizer.zero_grad()
            pred_delta = model(X)
            loss = criterion(pred_delta, y_delta, mask, last_frame, t_std, t_mean)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=grad_clip_max_norm)
            optimizer.step()
            ema.update(model)
            running_loss += loss.item()
        train_loss = running_loss / max(len(train_loader), 1)

        metrics_by_h = evaluate_direct_model_on_loader(model, val_loader, mask, horizons, t_mean, t_std)
        val_rmse_mean = float(np.mean([metrics_by_h[h]["RMSE"] for h in horizons]))
        logger.info(
            f"[TransferLearning:{gov_name}] epoch {epoch + 1}/{n_epochs} | train_loss={train_loss:.4f} | "
            + " | ".join(f"h{h} RMSE={metrics_by_h[h]['RMSE']:.4f}" for h in horizons)
            + f" | mean_RMSE={val_rmse_mean:.4f}"
        )

        if val_rmse_mean < best_val_rmse_mean:
            best_val_rmse_mean = val_rmse_mean
            best_metrics_by_h = metrics_by_h
            epochs_no_improve = 0
            save_checkpoint(
                ckpt_path, model, optimizer, None, ema, epoch, best_val_rmse_mean,
                extra={"hyperparams": hyperparams, "horizons": horizons, "lookback": lookback,
                       "frozen_modules": freeze_info["frozen_module_names"]},
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                logger.info(
                    f"[TransferLearning:{gov_name}] early stopping at epoch {epoch + 1} "
                    f"(no improvement in {patience} epochs)"
                )
                break

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not best_metrics_by_h:
        logger.warning(f"[TransferLearning:{gov_name}] best_metrics_by_h was never populated — running a fallback eval pass now.")
        best_metrics_by_h = evaluate_direct_model_on_loader(model, val_loader, mask, horizons, t_mean, t_std)

    ema_model = NileGuardDirectMultiHorizonModel(
        lookback=lookback, in_channels=ckpt["in_channels"],
        height=full_ds.H_gov, width=full_ds.W_gov, horizons=horizons,
        d_model=hyperparams.get("d_model", 64), nhead=hyperparams.get("nhead", 4),
        num_spatiotemporal_layers=hyperparams.get("num_layers", 2),
        dropout=hyperparams.get("dropout", 0.2), verbose=False,
    ).to(DEVICE)
    ema_model.load_state_dict(ema.shadow)
    ema_model.eval()

    improvement_by_h = {
        h: rmse_improvement_pct_safe(
            baseline_metrics_by_h[h]["RMSE"], best_metrics_by_h.get(h, {}).get("RMSE", np.nan),
            context=f"{gov_name}_transfer h{h}",
        )
        for h in horizons
    }
    logger.info(
        f"[TransferLearning:{gov_name}] FINE-TUNED FINAL - "
        + " | ".join(f"h{h} improvement_vs_baseline={improvement_by_h[h]:.2f}%" for h in horizons)
    )

    return {
        "governorate_id": governorate_id, "model": model, "ema_model": ema_model,
        "best_val_rmse_mean": best_val_rmse_mean, "best_metrics_by_h": best_metrics_by_h,
        "baseline_metrics_by_h": baseline_metrics_by_h, "rmse_improvement_by_h": improvement_by_h,
        "channel_mean": mean, "channel_std": std, "target_mask": full_ds.target_mask,
        "hyperparams": hyperparams, "lookback": lookback, "horizons": horizons,
        "bbox": ckpt["bbox"], "in_channels": ckpt["in_channels"], "freeze_info": freeze_info,
        "val_idx_usable": val_idx,
    }


# -----------------------------------------------------------------------
# 8.5 Run fine-tuning across all governorates
# -----------------------------------------------------------------------
fine_tuned_models_transfer = {}

for gov_id in range(1, N_GOVERNORATES + 1):
    gov_name = GOVERNORATE_NAMES[gov_id]
    try:
        result = fine_tune_fold_direct(
            governorate_id=gov_id,
            hyperparams=resolve_hyperparams(gov_id, default=DEFAULT_HYPERPARAMS_DIRECT),
        )
        fine_tuned_models_transfer[gov_id] = result
        logger.info(
            f"Fine-tuned (frozen backbone) - {gov_name}: "
            + " | ".join(f"h{h} improvement={result['rmse_improvement_by_h'][h]:.2f}%" for h in result["horizons"])
        )
    except FileNotFoundError as e:
        logger.warning(f"Skipping governorate {gov_name}: {e}")
    except Exception:
        logger.exception(f"Fine-tuning failed for governorate {gov_name} - skipping.")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _log_memory(f"after governorate {gov_name} (transfer fine-tuning)")


# -----------------------------------------------------------------------
# 8.6 Model Weight Export — single consolidated deployment file
# -----------------------------------------------------------------------
def export_transfer_deployment_weights(fine_tuned_models: dict, output_path=None) -> Path:
    """Consolidates every governorate's fine-tuned weights into one deployment file.

    Args:
        fine_tuned_models: ``{governorate_id: fine_tune_fold_direct(...) result}``.
        output_path: Destination `.pt` path (defaults to
            ``DEPLOYMENT_WEIGHTS_DIR / "NileGuard_transfer_production.pt"``).

    Returns:
        The `Path` the file was written to.
    """
    output_path = Path(output_path or (DEPLOYMENT_WEIGHTS_DIR / "NileGuard_transfer_production.pt"))
    output_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {}
    for gov_id, info in fine_tuned_models.items():
        gov_name = GOVERNORATE_NAMES[gov_id]
        payload[gov_name] = {
            "governorate_id": gov_id,
            "model_state": info["ema_model"].state_dict(),
            "lookback": info["lookback"],
            "horizons": info["horizons"],
            "hyperparams": info["hyperparams"],
            "channel_mean": info["channel_mean"],
            "channel_std": info["channel_std"],
            "target_mask": info["target_mask"],
            "bbox": info["bbox"],
            "in_channels": info["in_channels"],
            "feature_names": FEATURE_NAMES,
            "target_channel_index": TARGET_CHANNEL_INDEX,
            "gov_mask_channel_index": GOV_MASK_CHANNEL_INDEX,
            "architecture": "NileGuardDirectMultiHorizonModel_v2_transfer",
            "fine_tune_learning_rate": FINE_TUNE_LR,
            "frozen_modules": info["freeze_info"]["frozen_module_names"],
            "trainable_modules": info["freeze_info"]["trainable_module_names"],
            "validation_metrics_by_horizon": info["best_metrics_by_h"],
            "baseline_metrics_by_horizon": info["baseline_metrics_by_h"],
            "rmse_improvement_by_horizon": info["rmse_improvement_by_h"],
        }

    torch.save(payload, output_path)
    logger.info(
        f"[TransferLearning] Saved consolidated deployment weights -> {output_path} "
        f"({len(payload)} governorate model(s))"
    )
    return output_path


deployment_weights_path = export_transfer_deployment_weights(fine_tuned_models_transfer)


# -----------------------------------------------------------------------
# 8.7 CSV Data Export for Power BI
# -----------------------------------------------------------------------
# ASSUMPTION: TerraClimate coverage is documented elsewhere in this project
# as 1958-2021 monthly (64 years x 12 = 768 = T_TOTAL), so sample-start
# index 0 is assumed to correspond to 1958-01. Verify against the actual
# extraction date range before trusting `Forecast_Date` in a downstream
# dashboard; if the true start differs, only this one constant needs to change.
TERRACLIMATE_START_DATE = "1958-01-01"


def export_dashboard_predictions_csv(fine_tuned_models: dict, output_dir=None):
    """Exports tidy per-(governorate, horizon, date) predictions/actuals and a metrics summary, for Power BI.

    Predictions are governorate-mean values (averaged over each governorate's
    real pixels) so the export is one clean row per (governorate, horizon,
    forecast date) rather than one row per pixel.

    Args:
        fine_tuned_models: ``{governorate_id: fine_tune_fold_direct(...) result}``.
        output_dir: Output directory (defaults to `DASHBOARD_EXPORTS_DIR`).

    Returns:
        A ``(predictions_df, metrics_df)`` tuple of the two exported DataFrames.
    """
    output_dir = Path(output_dir or DASHBOARD_EXPORTS_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    date_index = pd.date_range(start=TERRACLIMATE_START_DATE, periods=T_TOTAL, freq="MS")

    prediction_rows = []
    metric_rows = []

    for gov_id, info in fine_tuned_models.items():
        gov_name = GOVERNORATE_NAMES[gov_id]
        lookback = info["lookback"]
        horizons = info["horizons"]
        mean, std = info["channel_mean"], info["channel_std"]
        t_mean, t_std = float(mean[TARGET_CHANNEL_INDEX]), float(std[TARGET_CHANNEL_INDEX])
        full_ds = NileGuardDirectMultiHorizonDataset(master_tensor, gov_id, lookback, mean, std, horizons=horizons)
        mask_np = info["target_mask"].numpy() if torch.is_tensor(info["target_mask"]) else info["target_mask"]
        model = info["ema_model"]

        val_idx = sorted(info["val_idx_usable"].tolist())
        if not val_idx:
            logger.warning(f"[DashboardExport] {gov_name}: no usable validation samples - skipping.")
            continue

        y_true_by_h, y_pred_by_h = _direct_model_predict(model, full_ds, val_idx, mask_np, t_mean, t_std, DEVICE)

        for h in horizons:
            y_true = y_true_by_h[h]   # (n_val_samples, n_pixels)
            y_pred = y_pred_by_h[h]   # (n_val_samples, n_pixels)

            metrics = compute_metrics(y_true, y_pred)
            metric_rows.append({
                "Governorate": gov_name, "Horizon_Months": h,
                "RMSE": metrics["RMSE"], "MAE": metrics["MAE"], "R2": metrics["R2"],
                "NSE": metrics["NSE"], "Bias": metrics["Bias"], "N_Samples": int(y_true.size),
            })

            y_true_per_sample = y_true.mean(axis=1)
            y_pred_per_sample = y_pred.mean(axis=1)

            for sample_i, sample_idx in enumerate(val_idx):
                target_month_idx = sample_idx + lookback - 1 + h
                target_date = date_index[target_month_idx] if target_month_idx < len(date_index) else pd.NaT
                prediction_rows.append({
                    "Governorate": gov_name,
                    "Horizon_Months": h,
                    "Forecast_Date": target_date.strftime("%Y-%m-%d") if pd.notna(target_date) else None,
                    "Actual_PDSI": float(y_true_per_sample[sample_i]),
                    "Predicted_PDSI": float(y_pred_per_sample[sample_i]),
                    "Error": float(y_pred_per_sample[sample_i] - y_true_per_sample[sample_i]),
                })

        logger.info(
            f"[DashboardExport] {gov_name}: "
            + " | ".join(f"h{h} RMSE={metric_rows[-len(horizons) + i]['RMSE']:.4f}" for i, h in enumerate(horizons))
        )

    predictions_df = pd.DataFrame(prediction_rows)
    if not predictions_df.empty:
        predictions_df = predictions_df.sort_values(
            ["Governorate", "Horizon_Months", "Forecast_Date"]
        ).reset_index(drop=True)

    metrics_df = pd.DataFrame(metric_rows)
    if not metrics_df.empty:
        metrics_df = metrics_df.sort_values(["Governorate", "Horizon_Months"]).reset_index(drop=True)

    predictions_path = output_dir / "nileguard_transfer_predictions_v2_transfer.csv"
    metrics_path = output_dir / "nileguard_transfer_metrics_v2_transfer.csv"
    predictions_df.to_csv(predictions_path, index=False)
    metrics_df.to_csv(metrics_path, index=False)

    logger.info(f"[DashboardExport] Predictions CSV -> {predictions_path} ({len(predictions_df)} rows)")
    logger.info(f"[DashboardExport] Metrics CSV -> {metrics_path} ({len(metrics_df)} rows)")

    return predictions_df, metrics_df


dashboard_predictions_df, dashboard_metrics_df = export_dashboard_predictions_csv(fine_tuned_models_transfer)

logger.info("=" * 78)
logger.info("Section 8 complete — transfer learning, deployment export, and Power BI CSVs done.")
logger.info(f"Deployment weights : {deployment_weights_path}")
logger.info(f"Dashboard exports  : {DASHBOARD_EXPORTS_DIR}")
logger.info("=" * 78)

dashboard_predictions_df.head(20)


2026-08-24 08:51:55 | INFO     | NileGuard | ==============================================================================
2026-08-24 08:51:55 | INFO     | NileGuard | NileGuard v2_optimized -> v2_transfer — Section 8: Transfer Learning
2026-08-24 08:51:55 | INFO     | NileGuard | ==============================================================================
2026-08-24 08:51:55 | INFO     | NileGuard | Pretrained artifacts source : results_v2_optimized\artifacts_v2_optimized
2026-08-24 08:51:55 | INFO     | NileGuard | Fine-tuning learning rate   : 0.0001
2026-08-24 08:51:55 | INFO     | NileGuard | Deployment weights output   : results_v2_transfer\deployment_weights
2026-08-24 08:51:55 | INFO     | NileGuard | Dashboard CSV output        : results_v2_transfer\dashboard_exports
2026-08-24 08:51:56 | INFO     | NileGuard | [TransferLearning] Loaded pretrained weights for Aswan <- results_v2_optimized\artifacts_v2_optimized\model_final_Aswan_v2_optimized.pth
2026-08-24 08:51:56 | INFO  

,Governorate,Horizon_Months,Forecast_Date,Actual_PDSI,Predicted_PDSI,Error
0,Aswan,1,2011-12-01,-0.112288,0.460889,0.573177
1,Aswan,1,2012-01-01,-0.413836,-0.116003,0.297833
2,Aswan,1,2012-02-01,-0.422752,-0.416341,0.006411
3,Aswan,1,2012-03-01,-0.478896,-0.423457,0.055439
4,Aswan,1,2012-04-01,-0.568706,-0.479120,0.089587
5,Aswan,1,2012-05-01,-0.488412,-0.569793,-0.081382
6,Aswan,1,2012-06-01,-0.418931,-0.488599,-0.069667
7,Aswan,1,2012-07-01,-0.359965,-0.420297,-0.060332
8,Aswan,1,2012-08-01,-0.312587,-0.360235,-0.047648
9,Aswan,1,2012-09-01,-0.271304,-0.312909,-0.041605
